# Laguna XS.2 — Falsification-Grade Causal Surgery
## v6 · clean experiment · RTX PRO 6000 Blackwell 96GB

This notebook is a **new experiment**, not an appended v5.

It is designed to resolve the confounds exposed by the previous run.

The central mistake in the previous comparison was treating:

\[
\text{same trainable parameter count}
\]

as equivalent to:

\[
\text{same effective learning opportunity}.
\]

For a sparse MoE those are not the same. An expert only receives useful
gradient on tokens that route through it.

v6 therefore tests **all of the following**:

1. routed-expert causal necessity;
2. routing frequency / routing mass;
3. effective supervised-token exposure;
4. initial gradient accessibility;
5. always-active shared-expert causality;
6. one-expert vs four-expert causal budgets;
7. natural routing vs forced equal-access training;
8. multiple training-order seeds;
9. multiple random baselines;
10. learning-rate sensitivity;
11. target improvement **and** signed target-specific gain;
12. surgery-forward equivalence and frozen-base integrity.

The decisive interpretations are:

```text
natural routing wins, forced exposure closes gap
    → routing accessibility caused the old result

routing wins even under forced exposure
    → causal necessity and plasticity are genuinely different

causal K=4 beats causal K=1
    → capability is distributed / coalition-level

shared expert wins at same parameter count
    → restricting search to routed experts was the wrong component assumption
```

## 1 — Install dependencies

In [2]:
# Keep the CUDA-enabled PyTorch build supplied by the instance.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2 — Runtime tuning for g7e.2xlarge

In [3]:
import os

# 8 vCPU host: leave headroom for Python / I/O.
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "4"

# 64 GiB RAM: conservative checkpoint-loading parallelism.
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

# CUDA.
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for AWS g7e.2xlarge.")

Runtime configured for AWS g7e.2xlarge.


## 3 — Hardware and storage preflight

In [4]:
import os
import shutil
import platform
from pathlib import Path

import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary or larger).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than a 64-GiB-class host detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen_devices = set()

for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

=== Host ===
Python: 3.10.12
Logical CPUs: 8
RAM total:     62.27 GiB
RAM available: 60.80 GiB

=== CUDA ===
Torch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
Compute capability: (12, 0)

=== Storage ===
Work root: /home/ec2-user/workspace
Disk free: 918.22 GiB
Hardware preflight: PASS


## 4 — Load the experiment CSV

Expected schema:

```text
split,kind,prompt,reference
```

Required minimum counts:

```text
selection / target   50
selection / control  50
train     / target   50
heldout   / target   50
heldout   / control  50
```

Lookup order:

1. `LAGUNA_EXPERIMENT_CSV`
2. `/home/ec2-user/workspace/laguna_frontend_experiment.csv`
3. `/workspace/laguna_frontend_experiment.csv`
4. `/mnt/data/laguna_frontend_experiment.csv`

In [5]:
import pandas as pd
import numpy as np

candidates = []

if os.environ.get("LAGUNA_EXPERIMENT_CSV"):
    candidates.append(
        Path(os.environ["LAGUNA_EXPERIMENT_CSV"]).expanduser()
    )

candidates.extend([
    Path("/home/ec2-user/workspace/laguna_frontend_experiment.csv"),
    Path("/workspace/laguna_frontend_experiment.csv"),
    Path("/mnt/data/laguna_frontend_experiment.csv"),
])

EXPERIMENT_CSV = next(
    (p.resolve() for p in candidates if p.exists()),
    None,
)

if EXPERIMENT_CSV is None:
    raise FileNotFoundError(
        "laguna_frontend_experiment.csv not found. "
        "Set LAGUNA_EXPERIMENT_CSV to the full file path."
    )

experiment_df = pd.read_csv(EXPERIMENT_CSV)

required_cols = {"split", "kind", "prompt", "reference"}
missing_cols = required_cols - set(experiment_df.columns)

if missing_cols:
    raise ValueError(
        f"Experiment CSV missing columns: {sorted(missing_cols)}"
    )

experiment_df["split"] = (
    experiment_df["split"].astype(str).str.lower().str.strip()
)
experiment_df["kind"] = (
    experiment_df["kind"].astype(str).str.lower().str.strip()
)

allowed_splits = {"selection", "train", "heldout"}
allowed_kinds = {"target", "control"}

bad_splits = sorted(set(experiment_df["split"]) - allowed_splits)
bad_kinds = sorted(set(experiment_df["kind"]) - allowed_kinds)

if bad_splits:
    raise ValueError(f"Unsupported split labels: {bad_splits}")

if bad_kinds:
    raise ValueError(f"Unsupported kind labels: {bad_kinds}")

minimums = {
    ("selection", "target"): 50,
    ("selection", "control"): 50,
    ("train", "target"): 50,
    ("heldout", "target"): 50,
    ("heldout", "control"): 50,
}

too_small = []

for (split, kind), minimum in minimums.items():
    actual = len(
        experiment_df[
            (experiment_df["split"] == split)
            & (experiment_df["kind"] == kind)
        ]
    )

    if actual < minimum:
        too_small.append((split, kind, actual, minimum))

if too_small:
    raise RuntimeError(
        "Dataset is below required minimums:\n"
        + "\n".join(
            f"{s}/{k}: {a} < {m}"
            for s, k, a, m in too_small
        )
    )

selection_df = experiment_df[
    experiment_df["split"] == "selection"
].reset_index(drop=True)

train_target_df = experiment_df[
    (experiment_df["split"] == "train")
    & (experiment_df["kind"] == "target")
].reset_index(drop=True)

heldout_df = experiment_df[
    experiment_df["split"] == "heldout"
].reset_index(drop=True)

print("Experiment CSV:", EXPERIMENT_CSV)

display(
    experiment_df.groupby(["split", "kind"])
    .size()
    .rename("count")
    .reset_index()
)

Experiment CSV: /home/ec2-user/workspace/laguna_frontend_experiment.csv


,split,kind,count
0,heldout,control,50
1,heldout,target,50
2,selection,control,50
3,selection,target,50
4,train,target,50


### Split leakage check

In [6]:
selection_prompts = set(selection_df["prompt"].astype(str))
train_prompts = set(train_target_df["prompt"].astype(str))
heldout_prompts = set(heldout_df["prompt"].astype(str))

leaks = {
    "selection_vs_train": selection_prompts & train_prompts,
    "selection_vs_heldout": selection_prompts & heldout_prompts,
    "train_vs_heldout": train_prompts & heldout_prompts,
}

for name, overlap in leaks.items():
    print(name, "overlap:", len(overlap))

if any(leaks.values()):
    raise RuntimeError("Prompt leakage detected across splits.")

print("Split leakage check: PASS")

selection_vs_train overlap: 0
selection_vs_heldout overlap: 0
train_vs_heldout overlap: 0
Split leakage check: PASS


## 5 — Resolve the official BF16 Laguna XS.2 checkpoint

In [7]:
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)

    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB."
        )

    print("Downloading Laguna XS.2 BF16...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.py",
            "*.jinja",
            "LICENSE*",
            "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]

if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print("Checkpoint verification: PASS")

MODEL_PATH: /home/ec2-user/workspace/models/Laguna-XS.2
14-shard payload: 66.889 GB
Checkpoint verification: PASS


/home/ec2-user/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6 — Register Laguna checkpoint conversion mapping

In [8]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose the native Laguna checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

Laguna checkpoint conversion mapping: REGISTERED
Conversion operations: 4


## 7 — Load BF16 model directly onto the RTX PRO 6000

In [9]:
import gc
import time
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if (
        ".mlp.experts" in k
        or ".mlp.gate" in k
        or "e_score_correction_bias" in k
    )
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")
print("BF16 load: PASS")

Transformers: 5.14.1


Loading weights: 100%|███████████████████████| 639/639 [08:26<00:00,  1.26it/s]


Loaded in 8.51 min
GPU allocated: 62.29 GiB
GPU reserved:  82.26 GiB
GPU peak:      63.29 GiB
Driver free:   12.17 GiB
Host RAM available: 58.74 GiB
BF16 load: PASS


## 8 — Validate Laguna MoE architecture

In [10]:
cfg = model.config

SPARSE_LAYERS = []

for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)

    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample_mlp = model.model.layers[SPARSE_LAYERS[0]].mlp

assert tuple(sample_mlp.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample_mlp.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample_mlp.experts.gate_up_proj[0].numel()
    + sample_mlp.experts.down_proj[0].numel()
)

print("Sparse layers:", SPARSE_LAYERS)
print(f"Params / expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("Architecture validation: PASS")

Sparse layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
Params / expert: 3,145,728 (3.146M)
Architecture validation: PASS


## 9 — Correct Laguna teacher-forcing format

The no-thinking assistant prefix is followed by a **newline** before the
reference answer:

```text
<assistant>
</think>
REFERENCE
```

In [11]:
def chat_prefix_text(prompt):
    messages = [{"role": "user", "content": prompt}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0

    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(
            f"Could not identify answer boundary for: {prompt!r}"
        )

    return full_ids, start

## 10 — Generic aligned scoring batch

In [12]:
def build_scoring_batch(df):
    df = df.reset_index(drop=True).copy()

    parsed = [
        parse_case(r.prompt, r.reference)
        for r in df.itertuples(index=False)
    ]

    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    targets = torch.full(
        (batch_size, max_ref),
        -100,
        dtype=torch.long,
    )

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1

        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

@torch.inference_mode()
def score_batch(batch):
    import torch.nn.functional as F

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    per_example = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example

    return result

## 11 — Build selection and held-out batches

In [13]:
SELECTION_BATCH = build_scoring_batch(selection_df)
HELDOUT_BATCH = build_scoring_batch(heldout_df)

SELECTION_BASE_NLL = score_batch(SELECTION_BATCH)
HELDOUT_BASE_NLL = score_batch(HELDOUT_BATCH)

selection_target_mask = selection_df["kind"].values == "target"
selection_control_mask = selection_df["kind"].values == "control"

heldout_target_mask = heldout_df["kind"].values == "target"
heldout_control_mask = heldout_df["kind"].values == "control"

print("Selection baseline target NLL :", float(SELECTION_BASE_NLL[selection_target_mask].mean()))
print("Selection baseline control NLL:", float(SELECTION_BASE_NLL[selection_control_mask].mean()))

print("Heldout baseline target NLL :", float(HELDOUT_BASE_NLL[heldout_target_mask].mean()))
print("Heldout baseline control NLL:", float(HELDOUT_BASE_NLL[heldout_control_mask].mean()))

Selection baseline target NLL : 5.2584381103515625
Selection baseline control NLL: 4.7993879318237305
Heldout baseline target NLL : 4.762903213500977
Heldout baseline control NLL: 4.120146751403809


## 12 — Scoring sanity check

In [14]:
demo_prefix = chat_prefix_text(selection_df.iloc[0]["prompt"])

print("Prefix tail:", repr(demo_prefix[-60:]))
print(
    "Teacher-forced boundary:",
    repr(
        (
            demo_prefix
            + "\n"
            + selection_df.iloc[0]["reference"]
        )[-90:]
    )
)

b = 0

ids = SELECTION_BATCH["input_ids"][b:b+1]
mask = SELECTION_BATCH["attention_mask"][b:b+1]
pos = SELECTION_BATCH["position_ids"][b:b+1]
keep = SELECTION_BATCH["pred_positions"]

with torch.inference_mode():
    selected_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=keep,
        return_dict=True,
    ).logits.float()

    full_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=0,
        return_dict=True,
    ).logits[:, keep, :].float()

max_diff = float(
    (selected_logits - full_logits).abs().max().item()
)

print("max |selective - full sliced logits|:", max_diff)

if max_diff > 1e-4:
    raise RuntimeError(
        "Selective-logit scorer does not match full-logit scoring."
    )

del selected_logits, full_logits
torch.cuda.empty_cache()

print("Scoring sanity: PASS")

Prefix tail: 'nk below its min-content width?\n</user>\n<assistant>\n</think>'
Teacher-forced boundary: 'ets a child shrink below its min-content width?\n</user>\n<assistant>\n</think>\nmin-width: 0;'
max |selective - full sliced logits|: 0.0
Scoring sanity: PASS


## 13 — Fixed-routing causal intervention

In [15]:
from contextlib import contextmanager
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)

    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")

    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(
    layer_idx,
    expert_ids=None,
    zero_all_routed=False,
    renormalize=False,
):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward

    expert_ids = (
        []
        if expert_ids is None
        else [int(x) for x in expert_ids]
    )

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = (
            original_forward(hidden_states)
        )

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )

            keep = ~torch.isin(selected_experts, ids)

            routing_weights = (
                routing_weights
                * keep.to(routing_weights.dtype)
            )

            if renormalize:
                denom = routing_weights.sum(
                    dim=-1,
                    keepdim=True,
                )

                routing_weights = torch.where(
                    denom > 0,
                    routing_weights
                    / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return (
            router_logits,
            routing_weights,
            selected_experts,
        )

    gate.forward = types.MethodType(
        patched_forward,
        gate,
    )

    try:
        yield
    finally:
        gate.forward = original_forward

## 14 — Causal score definition

In [16]:
CONTROL_PENALTY = 0.75

def summarize_selection_delta(ablated_nll):
    delta = np.asarray(ablated_nll) - SELECTION_BASE_NLL

    target_delta = float(
        delta[selection_target_mask].mean()
    )

    control_delta = float(
        delta[selection_control_mask].mean()
    )

    causal_specificity = (
        target_delta
        - CONTROL_PENALTY
        * max(control_delta, 0.0)
    )

    return {
        "target_delta_nll": target_delta,
        "control_delta_nll": control_delta,
        "causal_specificity": causal_specificity,
        "per_example_delta": delta,
    }

## 15 — Causal layer sweep

In [17]:
from tqdm.auto import tqdm
import time

RESULTS = WORK_ROOT / "laguna_xs2_v5_results"
RESULTS.mkdir(parents=True, exist_ok=True)

layer_rows = []

t0 = time.time()

for layer_idx in tqdm(
    SPARSE_LAYERS,
    desc="39-layer causal sweep",
):
    with gate_intervention(
        layer_idx,
        zero_all_routed=True,
    ):
        nll = score_batch(SELECTION_BATCH)

    m = summarize_selection_delta(nll)

    layer_rows.append({
        "layer": int(layer_idx),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

layer_df = pd.DataFrame(layer_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

layer_df.to_csv(
    RESULTS / "layer_causal_scores.csv",
    index=False,
)

print(
    f"Layer sweep time: {(time.time()-t0)/60:.2f} min"
)

display(layer_df.head(15))

39-layer causal sweep: 100%|███████████████████| 39/39 [00:18<00:00,  2.09it/s]

Layer sweep time: 0.31 min


,layer,target_delta_nll,control_delta_nll,causal_specificity
0,36,1.238812,0.218739,1.074758
1,38,0.351376,0.067112,0.301041
2,33,0.255101,0.082050,0.193563
3,27,0.240993,0.108592,0.159549
4,25,0.561571,0.564287,0.138355
5,24,0.061678,-0.067504,0.061678
6,16,0.189044,0.193727,0.043749
7,8,0.032251,0.011593,0.023556
8,18,0.056368,0.059783,0.011531
9,3,-0.029724,-0.082749,-0.029724


## 16 — Hierarchical expert search

In [18]:
def intervention_score(
    layer_idx,
    expert_ids,
    renormalize=False,
):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_batch(SELECTION_BATCH)

    return summarize_selection_delta(nll)

def hierarchical_expert_search(
    layer_idx,
    seed=17,
    initial_group_size=32,
    beam_width=3,
):
    rng = np.random.default_rng(seed)

    order = rng.permutation(
        cfg.num_experts
    ).tolist()

    frontier = [
        order[i:i+initial_group_size]
        for i in range(
            0,
            len(order),
            initial_group_size,
        )
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in tqdm(
            frontier,
            desc=f"L{layer_idx} level {level}",
            leave=False,
        ):
            m = intervention_score(
                layer_idx,
                group,
            )

            rec = {
                "layer": int(layer_idx),
                "seed": int(seed),
                "level": int(level),
                "group_size": len(group),
                "experts": list(map(int, group)),
                "target_delta_nll": m["target_delta_nll"],
                "control_delta_nll": m["control_delta_nll"],
                "causal_specificity": m["causal_specificity"],
            }

            history.append(rec)
            current.append(rec)

        current.sort(
            key=lambda x: x["causal_specificity"],
            reverse=True,
        )

        keep = current[:beam_width]

        if all(
            x["group_size"] == 1
            for x in keep
        ):
            break

        nxt = []

        for rec in keep:
            g = rec["experts"]

            if len(g) == 1:
                nxt.append(g)
            else:
                mid = len(g) // 2
                nxt.extend([
                    g[:mid],
                    g[mid:],
                ])

        frontier = [
            x for x in nxt if x
        ]

        level += 1

    hist = pd.DataFrame(history)

    leaf_size = hist["group_size"].min()

    leaves = hist[
        hist["group_size"] == leaf_size
    ].sort_values(
        "causal_specificity",
        ascending=False,
    )

    return hist, leaves

## 17 — Wider hierarchical search: top eight layers, four randomized partitions

In [19]:
TOP_LAYERS = 8
SEARCH_SEEDS = [17, 53, 101, 211]
INITIAL_GROUP_SIZE = 32
BEAM_WIDTH = 4

candidate_layers = (
    layer_df.head(TOP_LAYERS)["layer"]
    .astype(int)
    .tolist()
)

print("Candidate layers:", candidate_layers)

histories = []
leaf_frames = []

for layer_idx in candidate_layers:
    for seed in SEARCH_SEEDS:
        hist, leaves = hierarchical_expert_search(
            layer_idx,
            seed=seed,
            initial_group_size=INITIAL_GROUP_SIZE,
            beam_width=BEAM_WIDTH,
        )

        histories.append(hist)
        leaf_frames.append(leaves)

group_history = pd.concat(
    histories,
    ignore_index=True,
)

leaf_df = pd.concat(
    leaf_frames,
    ignore_index=True,
)

group_history.to_json(
    RESULTS / "hierarchical_group_history.json",
    orient="records",
    indent=2,
)

display(leaf_df.head(30))

Candidate layers: [36, 38, 33, 27, 25, 24, 16, 8]


,layer,seed,level,group_size,experts,target_delta_nll,control_delta_nll,causal_specificity
0,36,17,5,1,[229],1.280375e+00,4.886748e-01,0.913869
1,36,17,5,1,[95],1.858674e-02,-4.808240e-03,0.018587
2,36,17,5,1,[13],1.006799e-02,-3.219644e-03,0.010068
3,36,17,5,1,[245],7.967806e-04,-1.394809e-03,0.000797
4,36,17,5,1,[204],-9.918213e-07,7.629394e-08,-0.000001
5,36,17,5,1,[189],-3.347473e-03,-1.624335e-03,-0.003347
6,36,17,5,1,[254],-8.092509e-03,6.375506e-05,-0.008140
7,36,17,5,1,[130],-8.569057e-03,4.499732e-04,-0.008907
8,36,53,5,1,[229],1.280375e+00,4.886748e-01,0.913869
9,36,53,5,1,[95],1.858674e-02,-4.808240e-03,0.018587


## 18 — Exact individual expert validation

In [20]:
leaf_pairs = sorted({
    (int(r.layer), int(e))
    for r in leaf_df.itertuples(index=False)
    for e in r.experts
})

print("Unique leaf candidates:", len(leaf_pairs))

individual_rows = []

for layer_idx, expert_id in tqdm(
    leaf_pairs,
    desc="Individual validation",
):
    m = intervention_score(
        layer_idx,
        [expert_id],
    )

    individual_rows.append({
        "layer": int(layer_idx),
        "expert": int(expert_id),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"].tolist(),
    })

individual_df = pd.DataFrame(
    individual_rows
).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

display(individual_df.head(25))

Unique leaf candidates: 191


Individual validation: 100%|█████████████████| 191/191 [01:31<00:00,  2.10it/s]


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,per_example_delta
0,36,229,1.280375,0.488675,0.913869,"[-0.016780614852905273, 1.6728472709655762, 2...."
1,38,60,0.244515,0.049529,0.207369,"[0.0121612548828125, 0.32450056076049805, 1.86..."
2,27,146,0.236054,0.180448,0.100718,"[0.03569364547729492, -0.19040679931640625, 0...."
3,24,153,0.140403,0.079688,0.080637,"[-0.07714128494262695, -0.13550853729248047, -..."
4,16,59,0.075851,-0.013861,0.075851,"[0.32474350929260254, -0.13069605827331543, 0...."
5,33,166,0.070097,-0.011549,0.070097,"[0.011115074157714844, 0.03645682334899902, -0..."
6,24,206,0.138751,0.094239,0.068072,"[-0.06548047065734863, -0.1249685287475586, 0...."
7,24,146,0.067179,-0.000814,0.067179,"[-0.08603191375732422, -0.011610984802246094, ..."
8,33,206,0.058985,0.001513,0.057850,"[0.08012127876281738, -0.07839012145996094, -0..."
9,25,196,0.121510,0.086784,0.056422,"[-0.019655227661132812, 0.07716655731201172, -..."


## 19 — Bootstrap confidence intervals

In [21]:
def bootstrap_specificity(
    per_example_delta,
    n_boot=5000,
    seed=123,
):
    rng = np.random.default_rng(seed)

    d = np.asarray(
        per_example_delta,
        dtype=np.float64,
    )

    t = d[selection_target_mask]
    c = d[selection_control_mask]

    vals = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for i in range(n_boot):
        tb = rng.choice(
            t,
            size=len(t),
            replace=True,
        ).mean()

        cb = rng.choice(
            c,
            size=len(c),
            replace=True,
        ).mean()

        vals[i] = (
            tb
            - CONTROL_PENALTY
            * max(cb, 0.0)
        )

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

boot_rows = []

for r in individual_df.itertuples(index=False):
    boot_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        **bootstrap_specificity(
            r.per_example_delta
        ),
    })

boot_df = pd.DataFrame(boot_rows)

final_df = individual_df.merge(
    boot_df,
    on=["layer", "expert"],
    how="left",
).sort_values(
    ["p_positive", "causal_specificity"],
    ascending=False,
).reset_index(drop=True)

display(
    final_df[
        [
            "layer",
            "expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "p_positive",
        ]
    ].head(25)
)

,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,ci_2.5,ci_97.5,p_positive
0,36,229,1.280375,0.488675,0.913869,0.411105,1.435714,1.0000
1,38,60,0.244515,0.049529,0.207369,0.062659,0.400115,0.9994
2,33,166,0.070097,-0.011549,0.070097,0.026357,0.107441,0.9990
3,27,146,0.236054,0.180448,0.100718,0.031798,0.171312,0.9986
4,33,206,0.058985,0.001513,0.057850,0.013595,0.089700,0.9954
5,24,146,0.067179,-0.000814,0.067179,0.008492,0.110782,0.9870
6,33,234,0.044725,-0.011313,0.044725,-0.007953,0.094255,0.9450
7,27,192,0.046307,-0.008727,0.046307,-0.009265,0.091065,0.9442
8,24,206,0.138751,0.094239,0.068072,-0.018310,0.157196,0.9396
9,38,149,0.000906,0.000000,0.000906,-0.000094,0.002499,0.9094


## 20 — Renormalization robustness

In [22]:
ROBUST_TOP_N = min(
    16,
    len(final_df),
)

robust_rows = []

for r in tqdm(
    list(
        final_df.head(
            ROBUST_TOP_N
        ).itertuples(index=False)
    ),
    desc="Renormalized validation",
):
    m = intervention_score(
        int(r.layer),
        [int(r.expert)],
        renormalize=True,
    )

    robust_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        "renorm_target_delta_nll": m["target_delta_nll"],
        "renorm_control_delta_nll": m["control_delta_nll"],
        "renorm_causal_specificity": m["causal_specificity"],
    })

robust_df = pd.DataFrame(robust_rows)

final_robust = final_df.merge(
    robust_df,
    on=["layer", "expert"],
    how="left",
)

final_robust.drop(
    columns=["per_example_delta"],
).to_csv(
    RESULTS / "causal_candidates.csv",
    index=False,
)

display(
    final_robust[
        [
            "layer",
            "expert",
            "causal_specificity",
            "renorm_causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "p_positive",
        ]
    ].head(20)
)

Renormalized validation: 100%|█████████████████| 16/16 [00:07<00:00,  2.10it/s]


,layer,expert,causal_specificity,renorm_causal_specificity,ci_2.5,ci_97.5,p_positive
0,36,229,0.913869,0.904908,0.411105,1.435714,1.0000
1,38,60,0.207369,0.211055,0.062659,0.400115,0.9994
2,33,166,0.070097,0.080556,0.026357,0.107441,0.9990
3,27,146,0.100718,0.088627,0.031798,0.171312,0.9986
4,33,206,0.057850,0.032775,0.013595,0.089700,0.9954
5,24,146,0.067179,0.071523,0.008492,0.110782,0.9870
6,33,234,0.044725,0.020106,-0.007953,0.094255,0.9450
7,27,192,0.046307,0.042701,-0.009265,0.091065,0.9442
8,24,206,0.068072,0.069403,-0.018310,0.157196,0.9396
9,38,149,0.000906,-0.000836,-0.000094,0.002499,0.9094


## 21 — Strict causal expert selector

In [23]:
strict_candidates = final_robust[
    (final_robust["ci_2.5"] > 0)
    & (final_robust["causal_specificity"] > 0)
    & (final_robust["renorm_causal_specificity"] > 0)
].copy()

if strict_candidates.empty:
    raise RuntimeError(
        "No expert passed strict causal selection."
    )

strict_candidates = strict_candidates.sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

CAUSAL_SEED_PAIR = (
    int(strict_candidates.iloc[0]["layer"]),
    int(strict_candidates.iloc[0]["expert"]),
)

print("CAUSAL_SEED_PAIR =", CAUSAL_SEED_PAIR)

display(
    strict_candidates[
        [
            "layer",
            "expert",
            "causal_specificity",
            "ci_2.5",
            "ci_97.5",
            "renorm_causal_specificity",
        ]
    ].head(10)
)

CAUSAL_SEED_PAIR = (36, 229)


,layer,expert,causal_specificity,ci_2.5,ci_97.5,renorm_causal_specificity
0,36,229,0.913869,0.411105,1.435714,0.904908
1,38,60,0.207369,0.062659,0.400115,0.211055
2,27,146,0.100718,0.031798,0.171312,0.088627
3,33,166,0.070097,0.026357,0.107441,0.080556
4,24,146,0.067179,0.008492,0.110782,0.071523
5,33,206,0.057850,0.013595,0.089700,0.032775


## 22 — Global routing baseline on selection targets

In [24]:
@contextmanager
def capture_routing_for_batch(batch):
    records = {}
    originals = []

    valid_flat = (
        batch["attention_mask"]
        .reshape(-1)
        .bool()
    )

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward

        originals.append(
            (gate, original)
        )

        def make_forward(
            idx,
            original_forward,
        ):
            def patched(
                self,
                hidden_states,
            ):
                logits, weights, selected = (
                    original_forward(
                        hidden_states
                    )
                )

                with torch.no_grad():
                    mask = valid_flat

                    if mask.numel() == selected.shape[0]:
                        mask = mask.to(
                            selected.device
                        )
                    else:
                        mask = torch.ones(
                            selected.shape[0],
                            device=selected.device,
                            dtype=torch.bool,
                        )

                    ids = (
                        selected[mask]
                        .reshape(-1)
                        .long()
                    )

                    ws = (
                        weights[mask]
                        .reshape(-1)
                        .float()
                    )

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )

                    wsum.scatter_add_(
                        0,
                        ids,
                        ws,
                    )

                    records[int(idx)] = {
                        "tokens": int(
                            mask.sum().item()
                        ),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return (
                    logits,
                    weights,
                    selected,
                )

            return patched

        gate.forward = types.MethodType(
            make_forward(
                layer_idx,
                original,
            ),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

In [25]:
selection_target_df = selection_df[
    selection_df["kind"] == "target"
].reset_index(drop=True)

SELECTION_TARGET_BATCH = build_scoring_batch(
    selection_target_df
)

with capture_routing_for_batch(
    SELECTION_TARGET_BATCH
) as routing_records:
    _ = score_batch(
        SELECTION_TARGET_BATCH
    )

routing_rows = []

for layer_idx in SPARSE_LAYERS:
    rec = routing_records[int(layer_idx)]

    tokens = max(
        1,
        rec["tokens"],
    )

    for expert_id in range(
        cfg.num_experts
    ):
        routing_rows.append({
            "layer": int(layer_idx),
            "expert": int(expert_id),
            "selected_rate": float(
                rec["counts"][expert_id].item()
            ) / tokens,
            "routing_mass": float(
                rec["weight_sums"][expert_id].item()
            ) / tokens,
        })

global_routing_df = pd.DataFrame(
    routing_rows
).sort_values(
    ["routing_mass", "selected_rate"],
    ascending=False,
).reset_index(drop=True)

ROUTING_PAIR = (
    int(global_routing_df.iloc[0]["layer"]),
    int(global_routing_df.iloc[0]["expert"]),
)

print("ROUTING_PAIR =", ROUTING_PAIR)

display(
    global_routing_df.head(20)
)

global_routing_df.to_csv(
    RESULTS / "global_routing.csv",
    index=False,
)

ROUTING_PAIR = (29, 194)


,layer,expert,selected_rate,routing_mass
0,29,194,0.649721,0.188190
1,34,228,0.641042,0.163414
2,15,244,0.672350,0.160225
3,25,46,0.686609,0.146864
4,20,34,0.697148,0.142156
5,16,17,0.626782,0.122628
6,6,179,0.703038,0.115344
7,32,240,0.509609,0.098323
8,13,197,0.570986,0.097098
9,3,114,0.630502,0.092475


# Phase B — Effective routing exposure

The previous comparison matched **nominal parameters**, but it did not
measure how often those parameters were actually used on supervised tokens.

We measure routing on:

- selection target;
- selection control;
- train target;
- held-out target;
- held-out control.

For every set we record both:

- all valid sequence positions;
- **supervised positions** whose hidden states directly predict reference tokens.

In [26]:
def make_supervised_position_mask(batch):
    mask = torch.zeros_like(
        batch["attention_mask"],
        dtype=torch.bool,
    )

    valid_targets = batch["targets"].ne(-100)
    positions = batch["pred_positions"]

    for b in range(mask.shape[0]):
        valid = valid_targets[b]
        pos = positions[valid]
        mask[b, pos] = True

    return mask

def subset_batch(df):
    return build_scoring_batch(
        df.reset_index(drop=True)
    )

ROUTE_BATCHES = {
    "selection_target": subset_batch(
        selection_df[selection_df["kind"] == "target"]
    ),
    "selection_control": subset_batch(
        selection_df[selection_df["kind"] == "control"]
    ),
    "train_target": subset_batch(
        train_target_df
    ),
    "heldout_target": subset_batch(
        heldout_df[heldout_df["kind"] == "target"]
    ),
    "heldout_control": subset_batch(
        heldout_df[heldout_df["kind"] == "control"]
    ),
}

In [27]:
@contextmanager
def capture_routing_with_mask(batch, token_mask):
    records = {}
    originals = []

    token_mask_flat = (
        token_mask.reshape(-1).bool()
    )

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = (
                    original_forward(hidden_states)
                )

                with torch.no_grad():
                    mask = token_mask_flat

                    if mask.numel() != selected.shape[0]:
                        raise RuntimeError(
                            f"Routing mask/token mismatch at layer {idx}: "
                            f"{mask.numel()} vs {selected.shape[0]}"
                        )

                    mask = mask.to(selected.device)

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

In [28]:
def routing_stats_for_batch(name, batch):
    frames = []

    scopes = {
        "all_valid": batch["attention_mask"].bool(),
        "supervised": make_supervised_position_mask(batch),
    }

    for scope, token_mask in scopes.items():
        with capture_routing_with_mask(
            batch,
            token_mask,
        ) as records:
            _ = score_batch(batch)

        rows = []

        for layer_idx in SPARSE_LAYERS:
            rec = records[int(layer_idx)]
            tokens = max(1, rec["tokens"])

            for expert_id in range(cfg.num_experts):
                rows.append({
                    "dataset": name,
                    "scope": scope,
                    "layer": int(layer_idx),
                    "expert": int(expert_id),
                    "token_positions": int(rec["tokens"]),
                    "selected_hits": int(
                        rec["counts"][expert_id].item()
                    ),
                    "selected_rate": float(
                        rec["counts"][expert_id].item()
                    ) / tokens,
                    "routing_mass": float(
                        rec["weight_sums"][expert_id].item()
                    ) / tokens,
                })

        frames.append(pd.DataFrame(rows))

    return pd.concat(frames, ignore_index=True)

route_frames = []

for name, batch in ROUTE_BATCHES.items():
    print("Routing exposure:", name)
    route_frames.append(
        routing_stats_for_batch(name, batch)
    )

ROUTE_STATS = pd.concat(
    route_frames,
    ignore_index=True,
)

ROUTE_STATS.to_csv(
    RESULTS / "route_exposure_all_splits.csv",
    index=False,
)

print("Route exposure rows:", len(ROUTE_STATS))

Routing exposure: selection_target
Routing exposure: selection_control
Routing exposure: train_target
Routing exposure: heldout_target
Routing exposure: heldout_control
Route exposure rows: 99840


# Phase C — Shared-expert causal atlas

Laguna XS.2 has one always-active shared expert in every sparse MoE layer.
Its width is also 512, so its parameter count matches one routed expert:

\[
3 \times 2048 \times 512 = 3,145,728.
\]

That makes it an unusually clean matched-parameter control. Poolside's
configuration explicitly has a 512-wide shared expert and a routed-output
scaling factor; the shared expert is added to the routed branch by the MoE
block.

In [29]:
def get_shared_expert(layer_idx):
    mlp = get_sparse_mlp(layer_idx)

    for name in ("shared_expert", "shared_experts"):
        if hasattr(mlp, name):
            return getattr(mlp, name)

    raise AttributeError(
        f"No shared expert found at layer {layer_idx}."
    )

# Validate exact matched parameter count.
_shared = get_shared_expert(SPARSE_LAYERS[0])

shared_param_count = sum(
    p.numel()
    for p in _shared.parameters()
)

print("Routed expert params:", params_per_expert)
print("Shared expert params:", shared_param_count)

if shared_param_count != params_per_expert:
    raise RuntimeError(
        "Shared/routed parameter budgets are not identical."
    )

print("Shared-expert matched budget: PASS")

Routed expert params: 3145728
Shared expert params: 3145728
Shared-expert matched budget: PASS


In [30]:
@contextmanager
def zero_shared_expert(layer_idx):
    shared = get_shared_expert(layer_idx)
    original = shared.forward

    def patched(self, x):
        return torch.zeros_like(x)

    shared.forward = types.MethodType(
        patched,
        shared,
    )

    try:
        yield
    finally:
        shared.forward = original

shared_rows = []

for layer_idx in tqdm(
    SPARSE_LAYERS,
    desc="Shared-expert causal sweep",
):
    with zero_shared_expert(layer_idx):
        nll = score_batch(SELECTION_BATCH)

    m = summarize_selection_delta(nll)

    shared_rows.append({
        "layer": int(layer_idx),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

shared_causal_df = pd.DataFrame(
    shared_rows
).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

SHARED_CAUSAL_LAYER = int(
    shared_causal_df.iloc[0]["layer"]
)

shared_causal_df.to_csv(
    RESULTS / "shared_expert_causal_sweep.csv",
    index=False,
)

print(
    "Top causal shared expert layer:",
    SHARED_CAUSAL_LAYER,
)
display(shared_causal_df.head(15))

Shared-expert causal sweep: 100%|██████████████| 39/39 [00:18<00:00,  2.10it/s]

Top causal shared expert layer: 38


,layer,target_delta_nll,control_delta_nll,causal_specificity
0,38,3.178623,1.519338,2.039119
1,36,0.983593,0.966597,0.258645
2,35,0.797027,0.761508,0.225896
3,24,0.412779,0.346417,0.152966
4,27,0.175738,0.054893,0.134568
5,37,0.225700,0.134538,0.124797
6,23,0.317740,0.282073,0.106186
7,18,0.176197,0.114298,0.090473
8,21,0.210756,0.165567,0.086581
9,6,0.081194,-0.043983,0.081194


# Phase D — Cross-audit causal and routing candidates

The old search only exact-tested experts discovered by the causal hierarchy.
That means the routing winner could be outside the individually audited set.

v6 exact-tests the union of:

- top causal candidates;
- top globally routed candidates.

Every candidate gets:

- ordinary fixed-routing ablation;
- bootstrap CI;
- renormalized ablation;
- selection-target supervised routing exposure.

In [31]:
AUDIT_CAUSAL_TOP_N = min(20, len(final_robust))
AUDIT_ROUTING_TOP_N = 20

audit_pairs = set()

for r in final_robust.head(
    AUDIT_CAUSAL_TOP_N
).itertuples(index=False):
    audit_pairs.add(
        (int(r.layer), int(r.expert))
    )

for r in global_routing_df.head(
    AUDIT_ROUTING_TOP_N
).itertuples(index=False):
    audit_pairs.add(
        (int(r.layer), int(r.expert))
    )

audit_pairs = sorted(audit_pairs)

sel_route = ROUTE_STATS[
    (ROUTE_STATS["dataset"] == "selection_target")
    & (ROUTE_STATS["scope"] == "supervised")
].set_index(["layer", "expert"])

audit_rows = []

for layer_idx, expert_id in tqdm(
    audit_pairs,
    desc="Cross-method exact causal audit",
):
    m = intervention_score(
        layer_idx,
        [expert_id],
        renormalize=False,
    )

    mr = intervention_score(
        layer_idx,
        [expert_id],
        renormalize=True,
    )

    boot = bootstrap_specificity(
        m["per_example_delta"],
        n_boot=10000,
        seed=1000 + layer_idx * 257 + expert_id,
    )

    rr = sel_route.loc[
        (layer_idx, expert_id)
    ]

    audit_rows.append({
        "layer": int(layer_idx),
        "expert": int(expert_id),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "renorm_causal_specificity": mr["causal_specificity"],
        "ci_2.5": boot["ci_2.5"],
        "ci_97.5": boot["ci_97.5"],
        "p_positive": boot["p_positive"],
        "selection_supervised_selected_rate": float(
            rr["selected_rate"]
        ),
        "selection_supervised_routing_mass": float(
            rr["routing_mass"]
        ),
    })

audit_df = pd.DataFrame(
    audit_rows
).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

audit_df["hybrid_causal_exposure"] = (
    audit_df["causal_specificity"].clip(lower=0)
    * np.sqrt(
        audit_df[
            "selection_supervised_routing_mass"
        ].clip(lower=0)
        + 1e-12
    )
)

strict_audit = audit_df[
    (audit_df["ci_2.5"] > 0)
    & (audit_df["causal_specificity"] > 0)
    & (audit_df["renorm_causal_specificity"] > 0)
].copy()

if strict_audit.empty:
    raise RuntimeError(
        "No expert survived the cross-method strict causal audit."
    )

CAUSAL_PAIR = tuple(
    strict_audit.sort_values(
        "causal_specificity",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

HYBRID_PAIR = tuple(
    strict_audit.sort_values(
        "hybrid_causal_exposure",
        ascending=False,
    )
    .iloc[0][["layer", "expert"]]
    .astype(int)
    .tolist()
)

print("Initial hierarchical causal seed:", CAUSAL_SEED_PAIR)
print("Final cross-audited causal pair:", CAUSAL_PAIR)
print("Routing pair:", ROUTING_PAIR)
print("Causal × exposure pair:", HYBRID_PAIR)

audit_df.to_csv(
    RESULTS / "cross_method_causal_audit.csv",
    index=False,
)

display(audit_df.head(30))

Cross-method exact causal audit: 100%|█████████| 39/39 [00:44<00:00,  1.14s/it]

Initial hierarchical causal seed: (36, 229)
Final cross-audited causal pair: (36, 229)
Routing pair: (29, 194)
Causal × exposure pair: (36, 229)


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,renorm_causal_specificity,ci_2.5,ci_97.5,p_positive,selection_supervised_selected_rate,selection_supervised_routing_mass,hybrid_causal_exposure
0,36,229,1.280375,0.488675,0.913869,0.904908,0.416606,1.412502,1.0000,0.198758,0.052674,2.097399e-01
1,38,60,0.244515,0.049529,0.207369,0.211055,0.060338,0.393073,0.9993,0.347826,0.118510,7.138714e-02
2,27,146,0.236054,0.180448,0.100718,0.088627,0.031744,0.169095,0.9975,0.322981,0.068857,2.642892e-02
3,24,153,0.140403,0.079688,0.080637,0.070822,-0.042838,0.201996,0.9018,0.322981,0.051539,1.830636e-02
4,16,59,0.075851,-0.013861,0.075851,0.010672,-0.042467,0.179598,0.8999,0.459627,0.123212,2.662475e-02
5,33,166,0.070097,-0.011549,0.070097,0.080556,0.027203,0.107459,0.9985,0.285714,0.036918,1.346859e-02
6,24,206,0.138751,0.094239,0.068072,0.069403,-0.019081,0.154590,0.9382,0.310559,0.041173,1.381271e-02
7,24,146,0.067179,-0.000814,0.067179,0.071523,0.006112,0.110902,0.9855,0.440994,0.051916,1.530671e-02
8,33,206,0.058985,0.001513,0.057850,0.032775,0.014292,0.091060,0.9956,0.329193,0.048582,1.275097e-02
9,25,196,0.121510,0.086784,0.056422,0.056535,-0.027420,0.142575,0.9050,0.316770,0.038189,1.102606e-02


# Phase E — Trainable parameter banks and integrity checks

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import types


class SurgicalExpertBank(nn.Module):
    """
    Exact-baseline delta surgery.

    Forward =
        original_frozen_expert_output
        + trainable_selected_expert_output
        - frozen_selected_expert_output

    At initialization:
        trainable_selected == frozen_selected

    therefore:
        delta == 0 exactly

    and the untouched model's original expert kernel remains the base path.
    """

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(layer_idx), int(expert_id))
            for layer_idx, expert_id in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}

        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            # FP32 master parameters.
            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.params[down_key] = nn.Parameter(
                experts.down_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.key_map[
                (layer_idx, expert_id)
            ] = (
                gu_key,
                down_key,
            )

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        bank = self

        selected_layers = sorted({
            layer_idx
            for layer_idx, _ in self.selected_pairs
        })

        for layer_idx in selected_layers:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            original_forward = experts.forward

            self.original_forwards[
                layer_idx
            ] = original_forward

            selected_ids = sorted({
                expert_id
                for l, expert_id in self.selected_pairs
                if l == layer_idx
            })

            def make_forward(
                idx,
                base_experts,
                original_fn,
                selected_expert_ids,
            ):
                def patched_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    # -------------------------------------------------
                    # IMPORTANT:
                    # Preserve Laguna's ORIGINAL execution path.
                    # -------------------------------------------------
                    base_output = original_fn(
                        hidden_states,
                        top_k_index,
                        top_k_weights,
                    )

                    correction = torch.zeros_like(
                        base_output
                    )

                    for expert_id in selected_expert_ids:

                        # top_k_index:
                        # [num_tokens, top_k]
                        token_idx, top_k_pos = torch.where(
                            top_k_index == expert_id
                        )

                        if token_idx.numel() == 0:
                            continue

                        current_state = hidden_states[
                            token_idx
                        ]

                        compute_dtype = (
                            current_state.dtype
                        )

                        gu_key, down_key = (
                            bank.key_map[
                                (idx, expert_id)
                            ]
                        )

                        # ---------------------------------------------
                        # Trainable FP32 master -> current compute dtype
                        # Gradient propagates through .to(dtype).
                        # ---------------------------------------------
                        train_gu = bank.params[
                            gu_key
                        ].to(compute_dtype)

                        train_down = bank.params[
                            down_key
                        ].to(compute_dtype)

                        # ---------------------------------------------
                        # Frozen reference expert.
                        # ---------------------------------------------
                        frozen_gu = (
                            base_experts
                            .gate_up_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        frozen_down = (
                            base_experts
                            .down_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        # ---------------------------------------------
                        # Trainable selected expert.
                        # ---------------------------------------------
                        train_gate, train_up = F.linear(
                            current_state,
                            train_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        train_hidden = (
                            base_experts.act_fn(
                                train_gate
                            )
                            * train_up
                        )

                        train_hidden = F.linear(
                            train_hidden,
                            train_down,
                        )

                        # ---------------------------------------------
                        # Frozen selected expert using EXACT SAME
                        # manual computation as trainable branch.
                        #
                        # Therefore at initialization:
                        # train_hidden - frozen_hidden == 0.
                        # ---------------------------------------------
                        frozen_gate, frozen_up = F.linear(
                            current_state,
                            frozen_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        frozen_hidden = (
                            base_experts.act_fn(
                                frozen_gate
                            )
                            * frozen_up
                        )

                        frozen_hidden = F.linear(
                            frozen_hidden,
                            frozen_down,
                        )

                        route_weight = top_k_weights[
                            token_idx,
                            top_k_pos,
                            None,
                        ]

                        train_hidden = (
                            train_hidden
                            * route_weight
                        )

                        frozen_hidden = (
                            frozen_hidden
                            * route_weight
                        )

                        delta = (
                            train_hidden
                            - frozen_hidden
                        ).to(
                            base_output.dtype
                        )

                        # Out-of-place index_add preserves autograd.
                        correction = correction.index_add(
                            0,
                            token_idx,
                            delta,
                        )

                    return (
                        base_output
                        + correction
                    )

                return patched_forward

            experts.forward = types.MethodType(
                make_forward(
                    layer_idx,
                    experts,
                    original_forward,
                    selected_ids,
                ),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original_forward in (
            self.original_forwards.items()
        ):
            get_sparse_mlp(
                layer_idx
            ).experts.forward = (
                original_forward
            )

        self.original_forwards.clear()
        self.installed = False

In [33]:
def make_training_case(
    prompt,
    reference,
    max_length=1024,
):
    prefix_text = chat_prefix_text(
        prompt
    )

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(
            f"{len(full_ids)} tokens > {max_length}"
        )

    start = 0

    for a, b in zip(
        prefix_ids,
        full_ids,
    ):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError(
            "Could not identify answer boundary."
        )

    input_ids = torch.tensor(
        full_ids,
        dtype=torch.long,
        device="cuda:0",
    ).unsqueeze(0)

    attention_mask = torch.ones_like(
        input_ids
    )

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        dtype=torch.long,
        device="cuda:0",
    )

    targets = torch.tensor(
        full_ids[start:],
        dtype=torch.long,
        device="cuda:0",
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

MAX_TRAIN_TOKENS = 1024

TRAIN_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        MAX_TRAIN_TOKENS,
    )
    for r in train_target_df.itertuples(
        index=False
    )
]

print("Training examples:", len(TRAIN_CASES))
print(
    "Token lengths:",
    min(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
    "to",
    max(
        x["input_ids"].shape[1]
        for x in TRAIN_CASES
    ),
)

Training examples: 50
Token lengths: 66 to 83


In [34]:
SELECTION_TARGET_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        1024,
    )
    for r in selection_df[
        selection_df["kind"] == "target"
    ].itertuples(index=False)
]

In [35]:
class SurgicalSharedExpertBank(nn.Module):
    """
    Exact-baseline delta surgery for Laguna's shared expert.
    """

    def __init__(self, layer_idx):
        super().__init__()

        self.layer_idx = int(
            layer_idx
        )

        shared = get_shared_expert(
            self.layer_idx
        )

        self.gate_proj = nn.Parameter(
            shared.gate_proj.weight
            .detach()
            .float()
            .clone()
        )

        self.up_proj = nn.Parameter(
            shared.up_proj.weight
            .detach()
            .float()
            .clone()
        )

        self.down_proj = nn.Parameter(
            shared.down_proj.weight
            .detach()
            .float()
            .clone()
        )

        self.original_forward = None
        self.installed = False

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        shared = get_shared_expert(
            self.layer_idx
        )

        self.original_forward = (
            shared.forward
        )

        original_forward = (
            self.original_forward
        )

        bank = self
        act_fn = shared.act_fn

        def patched_forward(
            self_shared,
            x,
        ):
            # Keep original Laguna path intact.
            base_output = original_forward(
                x
            )

            dtype = x.dtype

            # Trainable copy.
            train_gate = F.linear(
                x,
                bank.gate_proj.to(dtype),
            )

            train_up = F.linear(
                x,
                bank.up_proj.to(dtype),
            )

            train_hidden = (
                act_fn(train_gate)
                * train_up
            )

            train_output = F.linear(
                train_hidden,
                bank.down_proj.to(dtype),
            )

            # Frozen reference using same manual operations.
            frozen_gate = F.linear(
                x,
                self_shared
                .gate_proj
                .weight
                .detach(),
            )

            frozen_up = F.linear(
                x,
                self_shared
                .up_proj
                .weight
                .detach(),
            )

            frozen_hidden = (
                act_fn(frozen_gate)
                * frozen_up
            )

            frozen_output = F.linear(
                frozen_hidden,
                self_shared
                .down_proj
                .weight
                .detach(),
            )

            correction = (
                train_output
                - frozen_output
            ).to(
                base_output.dtype
            )

            return (
                base_output
                + correction
            )

        shared.forward = types.MethodType(
            patched_forward,
            shared,
        )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        shared = get_shared_expert(
            self.layer_idx
        )

        shared.forward = (
            self.original_forward
        )

        self.original_forward = None
        self.installed = False

## Surgery equivalence tests — abort if patch changes the untrained model

In [36]:
def verify_routed_bank_equivalence(pair, tol=1e-5):
    probe_df = selection_df.head(8)
    batch = build_scoring_batch(probe_df)

    before = score_batch(batch)

    bank = SurgicalExpertBank([pair])
    bank.install()

    try:
        after = score_batch(batch)
    finally:
        bank.restore()
        del bank
        gc.collect()
        torch.cuda.empty_cache()

    diff = float(
        np.max(np.abs(before - after))
    )

    print(
        "Routed bank equivalence",
        pair,
        "max ΔNLL:",
        diff,
    )

    if diff > tol:
        raise RuntimeError(
            "SurgicalExpertBank changes outputs before training."
        )

def verify_shared_bank_equivalence(layer_idx, tol=1e-5):
    probe_df = selection_df.head(8)
    batch = build_scoring_batch(probe_df)

    before = score_batch(batch)

    bank = SurgicalSharedExpertBank(
        layer_idx
    )
    bank.install()

    try:
        after = score_batch(batch)
    finally:
        bank.restore()
        del bank
        gc.collect()
        torch.cuda.empty_cache()

    diff = float(
        np.max(np.abs(before - after))
    )

    print(
        "Shared bank equivalence L",
        layer_idx,
        "max ΔNLL:",
        diff,
    )

    if diff > tol:
        raise RuntimeError(
            "SurgicalSharedExpertBank changes outputs before training."
        )

for pair in {
    CAUSAL_PAIR,
    ROUTING_PAIR,
    HYBRID_PAIR,
}:
    verify_routed_bank_equivalence(pair)

verify_shared_bank_equivalence(
    SHARED_CAUSAL_LAYER
)

print("Surgery equivalence: PASS")

Routed bank equivalence (29, 194) max ΔNLL: 0.0
Routed bank equivalence (36, 229) max ΔNLL: 0.0
Shared bank equivalence L 38 max ΔNLL: 0.0
Surgery equivalence: PASS


# Phase F — Initial gradient accessibility

Causal necessity and trainability are different quantities.

We probe the **initial gradient norm** for every expert in the cross-audit
candidate pool using the same selection-target examples.

This is not allowed to touch held-out data.

We also normalize the raw gradient by routing exposure as a diagnostic.

In [37]:
def gradient_probe_pair(
    pair,
    cases,
    max_cases=8,
):
    pair = tuple(map(int, pair))

    bank = SurgicalExpertBank([pair])
    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    model.train()
    bank.train()

    for p in bank.parameters():
        p.grad = None

    used = min(
        max_cases,
        len(cases),
    )

    try:
        for case in cases[:used]:
            with torch.autocast(
                "cuda",
                dtype=torch.bfloat16,
            ):
                out = model(
                    input_ids=case["input_ids"],
                    attention_mask=case["attention_mask"],
                    use_cache=False,
                    logits_to_keep=case["pred_positions"],
                    return_dict=True,
                )

                logits = out.logits.float()

                loss = F.cross_entropy(
                    logits.reshape(
                        -1,
                        logits.shape[-1],
                    ),
                    case["targets"].reshape(-1),
                ) / used

            loss.backward()

            del out, logits, loss

        grad_sq = 0.0
        nonzero_tensors = 0

        for p in bank.parameters():
            if p.grad is not None:
                g = p.grad.detach().float()
                grad_sq += float(
                    torch.sum(g * g).item()
                )
                if torch.count_nonzero(g).item() > 0:
                    nonzero_tensors += 1

        return {
            "gradient_l2": float(
                np.sqrt(grad_sq)
            ),
            "nonzero_grad_tensors": int(
                nonzero_tensors
            ),
        }

    finally:
        bank.restore()

        try:
            model.gradient_checkpointing_disable()
        except Exception:
            pass

        if hasattr(
            model,
            "disable_input_require_grads",
        ):
            model.disable_input_require_grads()

        model.eval()

        del bank
        gc.collect()
        torch.cuda.empty_cache()

In [38]:
GRADIENT_PROBE_CASES = 8

gradient_rows = []

for r in tqdm(
    list(audit_df.itertuples(index=False)),
    desc="Candidate gradient accessibility",
):
    pair = (
        int(r.layer),
        int(r.expert),
    )

    g = gradient_probe_pair(
        pair,
        SELECTION_TARGET_CASES,
        max_cases=GRADIENT_PROBE_CASES,
    )

    rate = max(
        float(
            r.selection_supervised_selected_rate
        ),
        1e-8,
    )

    gradient_rows.append({
        "layer": pair[0],
        "expert": pair[1],
        "gradient_l2": g["gradient_l2"],
        "nonzero_grad_tensors": g[
            "nonzero_grad_tensors"
        ],
        "selection_supervised_selected_rate": float(
            r.selection_supervised_selected_rate
        ),
        "selection_supervised_routing_mass": float(
            r.selection_supervised_routing_mass
        ),
        "gradient_per_sqrt_selected_rate": (
            g["gradient_l2"]
            / np.sqrt(rate)
        ),
    })

gradient_df = pd.DataFrame(
    gradient_rows
).sort_values(
    "gradient_l2",
    ascending=False,
).reset_index(drop=True)

GRADIENT_PAIR = tuple(
    gradient_df.iloc[0][
        ["layer", "expert"]
    ]
    .astype(int)
    .tolist()
)

gradient_df.to_csv(
    RESULTS / "candidate_gradient_accessibility.csv",
    index=False,
)

print("GRADIENT_PAIR =", GRADIENT_PAIR)
display(gradient_df.head(30))

Candidate gradient accessibility: 100%|████████| 39/39 [02:45<00:00,  4.24s/it]

GRADIENT_PAIR = (25, 168)


,layer,expert,gradient_l2,nonzero_grad_tensors,selection_supervised_selected_rate,selection_supervised_routing_mass,gradient_per_sqrt_selected_rate
0,25,168,10.286367,2,0.310559,0.069973,18.458225
1,29,194,10.215510,2,0.347826,0.072311,17.321240
2,24,153,8.894264,2,0.322981,0.051539,15.650250
3,27,146,7.561443,2,0.322981,0.068857,13.305034
4,15,244,6.165009,2,0.403727,0.101933,9.702640
5,16,59,6.079509,2,0.459627,0.123212,8.967380
6,33,234,5.981252,2,0.310559,0.071071,10.732973
7,24,206,5.943840,2,0.310559,0.041173,10.665840
8,38,210,5.452672,2,0.397516,0.069174,8.648332
9,16,17,5.044159,2,0.329193,0.045474,8.791515


## Direct exposure audit for the deterministic K=1 selectors

In [39]:
K1_ROUTED_PAIRS = {
    "causal_routed": CAUSAL_PAIR,
    "routing_routed": ROUTING_PAIR,
    "hybrid_causal_exposure": HYBRID_PAIR,
    "gradient_routed": GRADIENT_PAIR,
}

exposure_rows = []

for selector, pair in K1_ROUTED_PAIRS.items():
    layer_idx, expert_id = pair

    for dataset in [
        "selection_target",
        "train_target",
        "heldout_target",
        "selection_control",
        "heldout_control",
    ]:
        for scope in [
            "supervised",
            "all_valid",
        ]:
            row = ROUTE_STATS[
                (ROUTE_STATS["dataset"] == dataset)
                & (ROUTE_STATS["scope"] == scope)
                & (ROUTE_STATS["layer"] == layer_idx)
                & (ROUTE_STATS["expert"] == expert_id)
            ].iloc[0]

            exposure_rows.append({
                "selector": selector,
                "layer": layer_idx,
                "expert": expert_id,
                "dataset": dataset,
                "scope": scope,
                "selected_hits": int(
                    row["selected_hits"]
                ),
                "token_positions": int(
                    row["token_positions"]
                ),
                "selected_rate": float(
                    row["selected_rate"]
                ),
                "routing_mass": float(
                    row["routing_mass"]
                ),
            })

selector_exposure_df = pd.DataFrame(
    exposure_rows
)

display(
    selector_exposure_df[
        (selector_exposure_df["dataset"] == "train_target")
        & (selector_exposure_df["scope"] == "supervised")
    ].sort_values(
        "routing_mass",
        ascending=False,
    )
)

selector_exposure_df.to_csv(
    RESULTS / "deterministic_selector_exposure.csv",
    index=False,
)

,selector,layer,expert,dataset,scope,selected_hits,token_positions,selected_rate,routing_mass
2,causal_routed,36,229,train_target,supervised,41,169,0.242604,0.071095
22,hybrid_causal_exposure,36,229,train_target,supervised,41,169,0.242604,0.071095
32,gradient_routed,25,168,train_target,supervised,50,169,0.295858,0.065800
12,routing_routed,29,194,train_target,supervised,53,169,0.313609,0.065278


# Phase G — K=4 parameter-budget scaling

A single expert can be too narrow. The causal method therefore also gets a
**coalition budget** of four experts.

Causal K=4 is selected greedily by **joint intervention score**, not by simply
taking the top four individual experts. This lets redundancy and synergy
affect selection.

Every K=4 method receives exactly:

\[
4 \times 3,145,728 = 12,582,912
\]

trainable routed-expert parameters.

In [40]:
@contextmanager
def multi_gate_intervention(
    pairs,
    renormalize=False,
):
    by_layer = {}

    for layer_idx, expert_id in pairs:
        by_layer.setdefault(
            int(layer_idx),
            [],
        ).append(int(expert_id))

    originals = []

    for layer_idx, expert_ids in by_layer.items():
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(
            original_forward,
            ids_to_zero,
        ):
            def patched(self, hidden_states):
                logits, weights, selected = (
                    original_forward(
                        hidden_states
                    )
                )

                ids = torch.tensor(
                    ids_to_zero,
                    device=selected.device,
                    dtype=selected.dtype,
                )

                keep = ~torch.isin(
                    selected,
                    ids,
                )

                weights = (
                    weights
                    * keep.to(weights.dtype)
                )

                if renormalize:
                    denom = weights.sum(
                        dim=-1,
                        keepdim=True,
                    )
                    weights = torch.where(
                        denom > 0,
                        weights
                        / denom.clamp_min(1e-12),
                        weights,
                    )

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(
                original,
                expert_ids,
            ),
            gate,
        )

    try:
        yield
    finally:
        for gate, original in originals:
            gate.forward = original

def group_causal_score(pairs):
    with multi_gate_intervention(
        pairs,
        renormalize=False,
    ):
        nll = score_batch(
            SELECTION_BATCH
        )

    return summarize_selection_delta(
        nll
    )

coalition_pool = [
    (int(r.layer), int(r.expert))
    for r in strict_audit.sort_values(
        "causal_specificity",
        ascending=False,
    ).head(12).itertuples(index=False)
]

if len(coalition_pool) < 4:
    raise RuntimeError(
        "Need at least four strict causal candidates for K=4."
    )

causal_k4 = []

while len(causal_k4) < 4:
    candidate_scores = []

    for pair in coalition_pool:
        if pair in causal_k4:
            continue

        proposed = causal_k4 + [pair]
        m = group_causal_score(proposed)

        candidate_scores.append(
            (m["causal_specificity"], pair)
        )

    candidate_scores.sort(
        reverse=True,
        key=lambda x: x[0],
    )

    chosen_score, chosen_pair = (
        candidate_scores[0]
    )

    causal_k4.append(chosen_pair)

    print(
        "Causal coalition step",
        len(causal_k4),
        "added",
        chosen_pair,
        "joint score",
        chosen_score,
    )

CAUSAL_K4 = causal_k4

ROUTING_K4 = [
    (int(r.layer), int(r.expert))
    for r in global_routing_df.head(4)
    .itertuples(index=False)
]

HYBRID_K4 = [
    (int(r.layer), int(r.expert))
    for r in strict_audit.sort_values(
        "hybrid_causal_exposure",
        ascending=False,
    ).head(4).itertuples(index=False)
]

GRADIENT_K4 = [
    (int(r.layer), int(r.expert))
    for r in gradient_df.head(4)
    .itertuples(index=False)
]

K4_SELECTORS = {
    "causal_k4": CAUSAL_K4,
    "routing_k4": ROUTING_K4,
    "hybrid_k4": HYBRID_K4,
    "gradient_k4": GRADIENT_K4,
}

for name, pairs in K4_SELECTORS.items():
    print(
        name,
        pairs,
        "params=",
        len(pairs) * params_per_expert,
    )

Causal coalition step 1 added (36, 229) joint score 0.9138692244887352
Causal coalition step 2 added (27, 146) joint score 1.014172539114952
Causal coalition step 3 added (33, 166) joint score 1.0959597080945969
Causal coalition step 4 added (24, 146) joint score 1.1534752249717712
causal_k4 [(36, 229), (27, 146), (33, 166), (24, 146)] params= 12582912
routing_k4 [(29, 194), (34, 228), (15, 244), (25, 46)] params= 12582912
hybrid_k4 [(36, 229), (38, 60), (27, 146), (24, 146)] params= 12582912
gradient_k4 [(25, 168), (29, 194), (24, 153), (27, 146)] params= 12582912


# Phase H — Forced-access training control

Natural-routing training confounds parameter quality with router access.

For the forced-access control, whenever the selected expert is absent from
top-k, v6 replaces the **lowest-weight selected expert ID** with the chosen
expert but **keeps that slot's routing weight unchanged**.

Therefore:

- top-k remains size 8;
- total routing-weight mass is unchanged;
- the selected expert receives a gradient path on every token;
- this modification exists only during training;
- held-out evaluation always restores normal routing.

If causal catches routing under this protocol, the old loss was mainly a
routing-accessibility problem.

In [41]:
from contextlib import nullcontext

@contextmanager
def force_selected_into_lowest_slots(
    selected_pairs,
):
    by_layer = {}

    for layer_idx, expert_id in selected_pairs:
        by_layer.setdefault(
            int(layer_idx),
            [],
        ).append(int(expert_id))

    originals = []

    for layer_idx, forced_ids in by_layer.items():
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(
            original_forward,
            ids_to_force,
        ):
            def patched(self, hidden_states):
                logits, weights, selected = (
                    original_forward(
                        hidden_states
                    )
                )

                selected = selected.clone()
                weights = weights.clone()

                protected = torch.zeros_like(
                    selected,
                    dtype=torch.bool,
                )

                for expert_id in ids_to_force:
                    present = (
                        selected == expert_id
                    ).any(dim=-1)

                    # Protect an already-present forced expert.
                    protected |= (
                        selected == expert_id
                    )

                    absent_rows = (
                        ~present
                    ).nonzero(
                        as_tuple=False
                    ).reshape(-1)

                    if absent_rows.numel() == 0:
                        continue

                    candidate_weights = (
                        weights[absent_rows]
                        .float()
                        .clone()
                    )

                    prot = protected[
                        absent_rows
                    ]

                    candidate_weights[
                        prot
                    ] = float("inf")

                    slots = candidate_weights.argmin(
                        dim=-1
                    )

                    selected[
                        absent_rows,
                        slots,
                    ] = int(expert_id)

                    protected[
                        absent_rows,
                        slots,
                    ] = True

                    # Intentionally keep the displaced
                    # slot's routing weight unchanged.

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(
                original,
                forced_ids,
            ),
            gate,
        )

    try:
        yield
    finally:
        for gate, original in originals:
            gate.forward = original

# Phase I — Evaluation metrics that do not hide broad general improvement

In [42]:
@torch.inference_mode()
def score_batch_detailed(batch):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    nll = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    preds = logits.argmax(dim=-1)

    correct = (
        preds.eq(targets)
        & valid
    )

    token_acc = (
        correct.sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    seq_exact = (
        (correct | ~valid)
        .all(dim=-1)
        .float()
    )

    result = {
        "nll": nll.cpu().numpy(),
        "token_acc": token_acc.cpu().numpy(),
        "seq_exact": seq_exact.cpu().numpy(),
    }

    del out, logits, losses, preds

    return result

HELDOUT_BASE_DETAIL = score_batch_detailed(
    HELDOUT_BATCH
)

def evaluate_current_model():
    d = score_batch_detailed(
        HELDOUT_BATCH
    )

    target_nll = float(
        d["nll"][
            heldout_target_mask
        ].mean()
    )
    control_nll = float(
        d["nll"][
            heldout_control_mask
        ].mean()
    )

    base_target_nll = float(
        HELDOUT_BASE_DETAIL["nll"][
            heldout_target_mask
        ].mean()
    )
    base_control_nll = float(
        HELDOUT_BASE_DETAIL["nll"][
            heldout_control_mask
        ].mean()
    )

    target_improvement = (
        base_target_nll
        - target_nll
    )
    control_improvement = (
        base_control_nll
        - control_nll
    )
    control_damage = (
        -control_improvement
    )

    # Utility: preserve general behavior, but does not
    # penalize broad improvements.
    utility_score = (
        target_improvement
        - CONTROL_PENALTY
        * max(control_damage, 0.0)
    )

    # Signed difference-in-differences:
    # how much more did target improve than control?
    specific_gain = (
        target_improvement
        - control_improvement
    )

    return {
        "heldout_target_nll": target_nll,
        "heldout_control_nll": control_nll,
        "target_improvement": float(
            target_improvement
        ),
        "control_improvement": float(
            control_improvement
        ),
        "control_damage": float(
            control_damage
        ),
        "utility_score": float(
            utility_score
        ),
        "specific_gain": float(
            specific_gain
        ),
        "target_token_acc": float(
            d["token_acc"][
                heldout_target_mask
            ].mean()
        ),
        "control_token_acc": float(
            d["token_acc"][
                heldout_control_mask
            ].mean()
        ),
        "target_seq_exact": float(
            d["seq_exact"][
                heldout_target_mask
            ].mean()
        ),
        "control_seq_exact": float(
            d["seq_exact"][
                heldout_control_mask
            ].mean()
        ),
        "per_example_nll": d["nll"],
    }

def bootstrap_adaptation(
    after_nll,
    n_boot=10000,
    seed=777,
):
    rng = np.random.default_rng(seed)

    after = np.asarray(
        after_nll,
        dtype=np.float64,
    )
    base_nll = np.asarray(
        HELDOUT_BASE_DETAIL["nll"],
        dtype=np.float64,
    )

    target_imp = (
        base_nll[heldout_target_mask]
        - after[heldout_target_mask]
    )
    control_imp = (
        base_nll[heldout_control_mask]
        - after[heldout_control_mask]
    )

    bt = np.empty(n_boot)
    bs = np.empty(n_boot)

    for i in range(n_boot):
        t = rng.choice(
            target_imp,
            size=len(target_imp),
            replace=True,
        ).mean()

        c = rng.choice(
            control_imp,
            size=len(control_imp),
            replace=True,
        ).mean()

        bt[i] = t
        bs[i] = t - c

    return {
        "target_ci_low": float(
            np.quantile(bt, 0.025)
        ),
        "target_ci_high": float(
            np.quantile(bt, 0.975)
        ),
        "specific_ci_low": float(
            np.quantile(bs, 0.025)
        ),
        "specific_ci_high": float(
            np.quantile(bs, 0.975)
        ),
    }

# Phase J — Training engine with frozen-base and parameter-delta audits

In [43]:
FIXED_LR = 1e-5
MATCHED_GRAD_ACCUM = 8
MATCHED_EPOCHS = 3
MATCHED_MAX_UPDATES = 50
MATCHED_WEIGHT_DECAY = 0.01

TRAIN_ORDER_SEEDS = [11, 23, 47]

RUN_NATURAL_K1 = True
RUN_FORCED_K1 = True
RUN_NATURAL_K4 = True
RUN_RANDOM_REPEATS = True
RUN_LR_SENSITIVITY = True

N_RANDOM_DRAWS = 5
LR_SENSITIVITY = [3e-6, 3e-5]

print("Training order seeds:", TRAIN_ORDER_SEEDS)

Training order seeds: [11, 23, 47]


In [44]:
def capture_routed_base_guard(pairs):
    guard = {}

    for layer_idx, expert_id in pairs:
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        guard[(layer_idx, expert_id)] = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu().clone(),
            experts.down_proj[
                expert_id
            ].detach().cpu().clone(),
        )

    return guard

def assert_routed_base_unchanged(guard):
    for (
        layer_idx,
        expert_id,
    ), (gu0, down0) in guard.items():
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        gu1 = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu()
        )
        down1 = (
            experts.down_proj[
                expert_id
            ].detach().cpu()
        )

        if not torch.equal(gu0, gu1):
            raise RuntimeError(
                f"Frozen base gate_up changed: "
                f"L{layer_idx}/E{expert_id}"
            )

        if not torch.equal(
            down0,
            down1,
        ):
            raise RuntimeError(
                f"Frozen base down changed: "
                f"L{layer_idx}/E{expert_id}"
            )

def state_l2(state):
    total = 0.0
    for v in state.values():
        x = v.detach().float()
        total += float(
            torch.sum(x * x).item()
        )
    return float(np.sqrt(total))

def state_delta_l2(before, after):
    total = 0.0

    for k in before:
        d = (
            after[k].detach().float()
            - before[k].detach().float()
        )
        total += float(
            torch.sum(d * d).item()
        )

    return float(np.sqrt(total))

In [45]:
def _configure_grad_checkpointing():
    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

def _disable_grad_checkpointing():
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    if hasattr(
        model,
        "disable_input_require_grads",
    ):
        model.disable_input_require_grads()

def train_routed_arm(
    selector,
    pairs,
    protocol,
    order_seed,
    lr=FIXED_LR,
    cases=None,
    epochs=MATCHED_EPOCHS,
    max_updates=MATCHED_MAX_UPDATES,
):
    pairs = [
        tuple(map(int, p))
        for p in pairs
    ]

    if cases is None:
        cases = TRAIN_CASES

    bank = SurgicalExpertBank(
        pairs
    )

    expected_params = (
        len(pairs)
        * params_per_expert
    )

    if (
        bank.trainable_parameter_count
        != expected_params
    ):
        raise RuntimeError(
            "Routed bank parameter budget mismatch."
        )

    guard = capture_routed_base_guard(
        pairs
    )

    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=float(lr),
        betas=(0.9, 0.95),
        weight_decay=MATCHED_WEIGHT_DECAY,
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    initial_state = {
        k: v.detach().cpu().clone()
        for k, v in bank.state_dict().items()
    }

    history = []
    raw_step = 0
    update_step = 0

    rng = np.random.default_rng(
        int(order_seed)
    )

    ctx = (
        force_selected_into_lowest_slots(
            pairs
        )
        if protocol == "forced_lowest_slot"
        else nullcontext()
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        with ctx:
            for epoch in range(
                int(epochs)
            ):
                order = rng.permutation(
                    len(cases)
                ).tolist()

                for position, case_idx in enumerate(
                    order
                ):
                    case = cases[
                        int(case_idx)
                    ]
                    raw_step += 1

                    with torch.autocast(
                        "cuda",
                        dtype=torch.bfloat16,
                    ):
                        out = model(
                            input_ids=case["input_ids"],
                            attention_mask=case["attention_mask"],
                            use_cache=False,
                            logits_to_keep=case["pred_positions"],
                            return_dict=True,
                        )

                        logits = (
                            out.logits.float()
                        )

                        loss = F.cross_entropy(
                            logits.reshape(
                                -1,
                                logits.shape[-1],
                            ),
                            case[
                                "targets"
                            ].reshape(-1),
                        )

                        scaled = (
                            loss
                            / MATCHED_GRAD_ACCUM
                        )

                    scaled.backward()

                    is_final = (
                        epoch == int(epochs) - 1
                        and position
                        == len(order) - 1
                    )

                    should_step = (
                        raw_step
                        % MATCHED_GRAD_ACCUM
                        == 0
                        or is_final
                    )

                    if should_step:
                        grad_norm = (
                            torch.nn.utils.clip_grad_norm_(
                                bank.parameters(),
                                1.0,
                            )
                        )

                        optimizer.step()
                        optimizer.zero_grad(
                            set_to_none=True
                        )

                        update_step += 1

                        history.append({
                            "selector": selector,
                            "budget_k": len(pairs),
                            "protocol": protocol,
                            "order_seed": int(
                                order_seed
                            ),
                            "lr": float(lr),
                            "update_step": int(
                                update_step
                            ),
                            "raw_step": int(
                                raw_step
                            ),
                            "loss": float(
                                loss.detach().item()
                            ),
                            "grad_norm": float(
                                grad_norm
                            ),
                        })

                    del out, logits, loss, scaled

                    if (
                        update_step
                        >= int(max_updates)
                    ):
                        break

                if (
                    update_step
                    >= int(max_updates)
                ):
                    break

        # Evaluation is ALWAYS natural routing.
        model.eval()
        bank.eval()

        metrics = evaluate_current_model()

        final_state = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        delta_l2 = state_delta_l2(
            initial_state,
            final_state,
        )
        base_l2 = state_l2(
            initial_state
        )

        boot = bootstrap_adaptation(
            metrics["per_example_nll"],
            seed=(
                int(order_seed)
                + len(pairs) * 1000
            ),
        )

        mean_grad = (
            float(
                np.mean([
                    h["grad_norm"]
                    for h in history
                ])
            )
            if history
            else 0.0
        )

        result = {
            "selector": selector,
            "module_type": "routed",
            "pairs": pairs,
            "budget_k": len(pairs),
            "protocol": protocol,
            "order_seed": int(order_seed),
            "lr": float(lr),
            "trainable_params": int(
                bank.trainable_parameter_count
            ),
            "updates": int(
                update_step
            ),
            "mean_grad_norm": mean_grad,
            "parameter_delta_l2": delta_l2,
            "relative_parameter_delta": (
                delta_l2
                / max(base_l2, 1e-12)
            ),
            "peak_gpu_gib": float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            **{
                k: v
                for k, v in metrics.items()
                if k != "per_example_nll"
            },
            **boot,
            "per_example_nll": metrics[
                "per_example_nll"
            ],
            "history": history,
            "state_dict_cpu": final_state,
        }

        return result

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        assert_routed_base_unchanged(
            guard
        )

        del optimizer
        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()

In [46]:
def capture_shared_base_guard(layer_idx):
    shared = get_shared_expert(
        layer_idx
    )

    return {
        "gate": shared.gate_proj.weight
        .detach().cpu().clone(),
        "up": shared.up_proj.weight
        .detach().cpu().clone(),
        "down": shared.down_proj.weight
        .detach().cpu().clone(),
    }

def assert_shared_base_unchanged(
    layer_idx,
    guard,
):
    shared = get_shared_expert(
        layer_idx
    )

    current = {
        "gate": shared.gate_proj.weight
        .detach().cpu(),
        "up": shared.up_proj.weight
        .detach().cpu(),
        "down": shared.down_proj.weight
        .detach().cpu(),
    }

    for k in guard:
        if not torch.equal(
            guard[k],
            current[k],
        ):
            raise RuntimeError(
                f"Frozen shared base changed: "
                f"L{layer_idx}/{k}"
            )

def train_shared_arm(
    selector,
    layer_idx,
    order_seed,
    lr=FIXED_LR,
):
    layer_idx = int(layer_idx)

    bank = SurgicalSharedExpertBank(
        layer_idx
    )

    if (
        bank.trainable_parameter_count
        != params_per_expert
    ):
        raise RuntimeError(
            "Shared expert parameter budget mismatch."
        )

    guard = capture_shared_base_guard(
        layer_idx
    )

    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=float(lr),
        betas=(0.9, 0.95),
        weight_decay=MATCHED_WEIGHT_DECAY,
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    initial_state = {
        k: v.detach().cpu().clone()
        for k, v in bank.state_dict().items()
    }

    history = []
    raw_step = 0
    update_step = 0
    rng = np.random.default_rng(
        int(order_seed)
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(
            MATCHED_EPOCHS
        ):
            order = rng.permutation(
                len(TRAIN_CASES)
            ).tolist()

            for position, case_idx in enumerate(
                order
            ):
                case = TRAIN_CASES[
                    int(case_idx)
                ]
                raw_step += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case["input_ids"],
                        attention_mask=case["attention_mask"],
                        use_cache=False,
                        logits_to_keep=case["pred_positions"],
                        return_dict=True,
                    )

                    logits = (
                        out.logits.float()
                    )

                    loss = F.cross_entropy(
                        logits.reshape(
                            -1,
                            logits.shape[-1],
                        ),
                        case[
                            "targets"
                        ].reshape(-1),
                    )

                    scaled = (
                        loss
                        / MATCHED_GRAD_ACCUM
                    )

                scaled.backward()

                is_final = (
                    epoch
                    == MATCHED_EPOCHS - 1
                    and position
                    == len(order) - 1
                )

                should_step = (
                    raw_step
                    % MATCHED_GRAD_ACCUM
                    == 0
                    or is_final
                )

                if should_step:
                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            bank.parameters(),
                            1.0,
                        )
                    )

                    optimizer.step()
                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "selector": selector,
                        "budget_k": 1,
                        "protocol": "always_active_shared",
                        "order_seed": int(
                            order_seed
                        ),
                        "lr": float(lr),
                        "update_step": int(
                            update_step
                        ),
                        "raw_step": int(
                            raw_step
                        ),
                        "loss": float(
                            loss.detach().item()
                        ),
                        "grad_norm": float(
                            grad_norm
                        ),
                    })

                del out, logits, loss, scaled

                if (
                    update_step
                    >= MATCHED_MAX_UPDATES
                ):
                    break

            if (
                update_step
                >= MATCHED_MAX_UPDATES
            ):
                break

        model.eval()
        bank.eval()

        metrics = evaluate_current_model()

        final_state = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        delta_l2 = state_delta_l2(
            initial_state,
            final_state,
        )
        base_l2 = state_l2(
            initial_state
        )

        boot = bootstrap_adaptation(
            metrics["per_example_nll"],
            seed=int(order_seed) + 9000,
        )

        result = {
            "selector": selector,
            "module_type": "shared",
            "pairs": [(layer_idx, "shared")],
            "budget_k": 1,
            "protocol": "always_active_shared",
            "order_seed": int(order_seed),
            "lr": float(lr),
            "trainable_params": int(
                bank.trainable_parameter_count
            ),
            "updates": int(
                update_step
            ),
            "mean_grad_norm": float(
                np.mean([
                    h["grad_norm"]
                    for h in history
                ])
            ) if history else 0.0,
            "parameter_delta_l2": state_delta_l2(
                initial_state,
                final_state,
            ),
            "relative_parameter_delta": (
                delta_l2
                / max(base_l2, 1e-12)
            ),
            "peak_gpu_gib": float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            **{
                k: v
                for k, v in metrics.items()
                if k != "per_example_nll"
            },
            **boot,
            "per_example_nll": metrics[
                "per_example_nll"
            ],
            "history": history,
            "state_dict_cpu": final_state,
        }

        return result

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        assert_shared_base_unchanged(
            layer_idx,
            guard,
        )

        del optimizer
        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()

# Phase K — Repeated random baselines

In [47]:
rng = np.random.default_rng(2026)

all_pairs = [
    (int(layer_idx), int(expert_id))
    for layer_idx in SPARSE_LAYERS
    for expert_id in range(cfg.num_experts)
]

deterministic_pairs = set(
    K1_ROUTED_PAIRS.values()
)

available_global = [
    p for p in all_pairs
    if p not in deterministic_pairs
]

global_idx = rng.choice(
    len(available_global),
    size=N_RANDOM_DRAWS,
    replace=False,
)

RANDOM_GLOBAL_PAIRS = [
    available_global[int(i)]
    for i in global_idx
]

same_layer_pool = [
    (CAUSAL_PAIR[0], e)
    for e in range(cfg.num_experts)
    if (
        CAUSAL_PAIR[0],
        e,
    ) not in deterministic_pairs
]

same_idx = rng.choice(
    len(same_layer_pool),
    size=N_RANDOM_DRAWS,
    replace=False,
)

RANDOM_SAME_LAYER_PAIRS = [
    same_layer_pool[int(i)]
    for i in same_idx
]

print(
    "Random global:",
    RANDOM_GLOBAL_PAIRS,
)
print(
    "Random same layer:",
    RANDOM_SAME_LAYER_PAIRS,
)

Random global: [(25, 243), (15, 63), (7, 249), (34, 52), (2, 7)]
Random same layer: [(36, 231), (36, 209), (36, 200), (36, 89), (36, 178)]


# Phase L — Run the falsification matrix

Main deterministic K=1 natural-routing arms:

- causal routed;
- routing routed;
- causal×exposure hybrid;
- gradient-accessible routed;
- top causal shared expert.

Then:

- causal vs routing under forced access;
- K=4 causal coalition vs K=4 routing/hybrid/gradient;
- repeated random controls;
- LR sensitivity for causal vs routing.

All held-out evaluation is performed under **normal routing**.

In [48]:
RUN_RESULTS = []
RUN_HISTORY = []
RUN_STATES = {}

def record_result(result):
    RUN_RESULTS.append({
        k: v
        for k, v in result.items()
        if k not in {
            "per_example_nll",
            "history",
            "state_dict_cpu",
        }
    })

    RUN_HISTORY.extend(
        result["history"]
    )

    key = (
        result["selector"],
        result["budget_k"],
        result["protocol"],
        result["order_seed"],
        result["lr"],
    )

    RUN_STATES[key] = {
        "state_dict_cpu": result[
            "state_dict_cpu"
        ],
        "per_example_nll": result[
            "per_example_nll"
        ],
    }

    print(
        f"{result['selector']:28s} "
        f"K={result['budget_k']} "
        f"{result['protocol']:20s} "
        f"seed={result['order_seed']} "
        f"lr={result['lr']:.1e} "
        f"target={result['target_improvement']:+.4f} "
        f"control_imp={result['control_improvement']:+.4f} "
        f"specific={result['specific_gain']:+.4f}"
    )

In [49]:
if RUN_NATURAL_K1:
    for seed in TRAIN_ORDER_SEEDS:
        for selector, pair in K1_ROUTED_PAIRS.items():
            print(
                "\nNATURAL K1",
                selector,
                pair,
                "seed",
                seed,
            )

            r = train_routed_arm(
                selector=selector,
                pairs=[pair],
                protocol="natural",
                order_seed=seed,
                lr=FIXED_LR,
            )
            record_result(r)

        print(
            "\nNATURAL K1 causal shared",
            SHARED_CAUSAL_LAYER,
            "seed",
            seed,
        )

        r = train_shared_arm(
            selector="causal_shared",
            layer_idx=SHARED_CAUSAL_LAYER,
            order_seed=seed,
            lr=FIXED_LR,
        )
        record_result(r)


NATURAL K1 causal_routed (36, 229) seed 11
causal_routed                K=1 natural              seed=11 lr=1.0e-05 target=+0.0306 control_imp=+0.0283 specific=+0.0023

NATURAL K1 routing_routed (29, 194) seed 11
routing_routed               K=1 natural              seed=11 lr=1.0e-05 target=+0.7899 control_imp=+0.6711 specific=+0.1188

NATURAL K1 hybrid_causal_exposure (36, 229) seed 11
hybrid_causal_exposure       K=1 natural              seed=11 lr=1.0e-05 target=+0.0306 control_imp=+0.0283 specific=+0.0023

NATURAL K1 gradient_routed (25, 168) seed 11
gradient_routed              K=1 natural              seed=11 lr=1.0e-05 target=+1.0969 control_imp=+0.8024 specific=+0.2944

NATURAL K1 causal shared 38 seed 11
causal_shared                K=1 always_active_shared seed=11 lr=1.0e-05 target=+0.6204 control_imp=+0.5168 specific=+0.1035

NATURAL K1 causal_routed (36, 229) seed 23
causal_routed                K=1 natural              seed=23 lr=1.0e-05 target=+0.0291 control_imp=+0.030

In [50]:
if RUN_FORCED_K1:
    for seed in TRAIN_ORDER_SEEDS:
        for selector, pair in {
            "causal_routed": CAUSAL_PAIR,
            "routing_routed": ROUTING_PAIR,
        }.items():
            print(
                "\nFORCED K1",
                selector,
                pair,
                "seed",
                seed,
            )

            r = train_routed_arm(
                selector=selector,
                pairs=[pair],
                protocol="forced_lowest_slot",
                order_seed=seed,
                lr=FIXED_LR,
            )
            record_result(r)


FORCED K1 causal_routed (36, 229) seed 11
causal_routed                K=1 forced_lowest_slot   seed=11 lr=1.0e-05 target=+0.0227 control_imp=+0.0188 specific=+0.0039

FORCED K1 routing_routed (29, 194) seed 11
routing_routed               K=1 forced_lowest_slot   seed=11 lr=1.0e-05 target=+0.7939 control_imp=+0.6662 specific=+0.1277

FORCED K1 causal_routed (36, 229) seed 23
causal_routed                K=1 forced_lowest_slot   seed=23 lr=1.0e-05 target=+0.0231 control_imp=+0.0225 specific=+0.0006

FORCED K1 routing_routed (29, 194) seed 23
routing_routed               K=1 forced_lowest_slot   seed=23 lr=1.0e-05 target=+0.7928 control_imp=+0.6563 specific=+0.1365

FORCED K1 causal_routed (36, 229) seed 47
causal_routed                K=1 forced_lowest_slot   seed=47 lr=1.0e-05 target=+0.0212 control_imp=+0.0193 specific=+0.0019

FORCED K1 routing_routed (29, 194) seed 47
routing_routed               K=1 forced_lowest_slot   seed=47 lr=1.0e-05 target=+0.7869 control_imp=+0.6642 specif

In [51]:
if RUN_NATURAL_K4:
    for seed in TRAIN_ORDER_SEEDS:
        for selector, pairs in K4_SELECTORS.items():
            print(
                "\nNATURAL K4",
                selector,
                pairs,
                "seed",
                seed,
            )

            r = train_routed_arm(
                selector=selector,
                pairs=pairs,
                protocol="natural",
                order_seed=seed,
                lr=FIXED_LR,
            )
            record_result(r)


NATURAL K4 causal_k4 [(36, 229), (27, 146), (33, 166), (24, 146)] seed 11
causal_k4                    K=4 natural              seed=11 lr=1.0e-05 target=+1.0641 control_imp=+0.8661 specific=+0.1981

NATURAL K4 routing_k4 [(29, 194), (34, 228), (15, 244), (25, 46)] seed 11
routing_k4                   K=4 natural              seed=11 lr=1.0e-05 target=+1.8961 control_imp=+1.3781 specific=+0.5180

NATURAL K4 hybrid_k4 [(36, 229), (38, 60), (27, 146), (24, 146)] seed 11
hybrid_k4                    K=4 natural              seed=11 lr=1.0e-05 target=+1.0578 control_imp=+0.8151 specific=+0.2427

NATURAL K4 gradient_k4 [(25, 168), (29, 194), (24, 153), (27, 146)] seed 11
gradient_k4                  K=4 natural              seed=11 lr=1.0e-05 target=+2.7204 control_imp=+2.1214 specific=+0.5991

NATURAL K4 causal_k4 [(36, 229), (27, 146), (33, 166), (24, 146)] seed 23
causal_k4                    K=4 natural              seed=23 lr=1.0e-05 target=+1.0621 control_imp=+0.8557 specific=+0.2064

In [52]:
if RUN_RANDOM_REPEATS:
    random_seed = TRAIN_ORDER_SEEDS[0]

    for i, pair in enumerate(
        RANDOM_GLOBAL_PAIRS
    ):
        r = train_routed_arm(
            selector=f"random_global_{i}",
            pairs=[pair],
            protocol="natural",
            order_seed=random_seed,
            lr=FIXED_LR,
        )
        record_result(r)

    for i, pair in enumerate(
        RANDOM_SAME_LAYER_PAIRS
    ):
        r = train_routed_arm(
            selector=f"random_same_layer_{i}",
            pairs=[pair],
            protocol="natural",
            order_seed=random_seed,
            lr=FIXED_LR,
        )
        record_result(r)

random_global_0              K=1 natural              seed=11 lr=1.0e-05 target=-0.0151 control_imp=+0.0107 specific=-0.0258
random_global_1              K=1 natural              seed=11 lr=1.0e-05 target=+0.0500 control_imp=+0.0325 specific=+0.0175
random_global_2              K=1 natural              seed=11 lr=1.0e-05 target=+0.0000 control_imp=+0.0000 specific=+0.0000
random_global_3              K=1 natural              seed=11 lr=1.0e-05 target=+0.0049 control_imp=+0.0029 specific=+0.0020
random_global_4              K=1 natural              seed=11 lr=1.0e-05 target=+0.0226 control_imp=+0.0427 specific=-0.0201
random_same_layer_0          K=1 natural              seed=11 lr=1.0e-05 target=+0.0050 control_imp=+0.0051 specific=-0.0001
random_same_layer_1          K=1 natural              seed=11 lr=1.0e-05 target=+0.0024 control_imp=-0.0036 specific=+0.0061
random_same_layer_2          K=1 natural              seed=11 lr=1.0e-05 target=+0.0039 control_imp=+0.0055 specific=-0.0017


In [53]:
if RUN_LR_SENSITIVITY:
    sensitivity_seed = (
        TRAIN_ORDER_SEEDS[0]
    )

    for lr in LR_SENSITIVITY:
        for selector, pair in {
            "causal_routed_lr_sensitivity": CAUSAL_PAIR,
            "routing_routed_lr_sensitivity": ROUTING_PAIR,
        }.items():
            r = train_routed_arm(
                selector=selector,
                pairs=[pair],
                protocol="natural_lr_sensitivity",
                order_seed=sensitivity_seed,
                lr=lr,
            )
            record_result(r)

causal_routed_lr_sensitivity K=1 natural_lr_sensitivity seed=11 lr=3.0e-06 target=+0.0136 control_imp=-0.0003 specific=+0.0139
routing_routed_lr_sensitivity K=1 natural_lr_sensitivity seed=11 lr=3.0e-06 target=+0.1302 control_imp=+0.1146 specific=+0.0156
causal_routed_lr_sensitivity K=1 natural_lr_sensitivity seed=11 lr=3.0e-05 target=+0.0869 control_imp=+0.0731 specific=+0.0138
routing_routed_lr_sensitivity K=1 natural_lr_sensitivity seed=11 lr=3.0e-05 target=+2.1436 control_imp=+1.6427 specific=+0.5009


# Phase M — Aggregate results across seeds and baselines

In [54]:
runs_df = pd.DataFrame(
    RUN_RESULTS
)

history_df = pd.DataFrame(
    RUN_HISTORY
)

display(
    runs_df.sort_values(
        [
            "budget_k",
            "protocol",
            "target_improvement",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

runs_df.to_csv(
    RESULTS / "v6_all_training_runs.csv",
    index=False,
)

history_df.to_csv(
    RESULTS / "v6_training_history.csv",
    index=False,
)

,selector,module_type,pairs,budget_k,protocol,order_seed,lr,trainable_params,updates,mean_grad_norm,...,utility_score,specific_gain,target_token_acc,control_token_acc,target_seq_exact,control_seq_exact,target_ci_low,target_ci_high,specific_ci_low,specific_ci_high
9,causal_shared,shared,"[(38, shared)]",1,always_active_shared,23,0.000010,3145728,19,10.926318,...,0.626537,0.120088,0.621286,0.639667,0.00,0.00,0.554548,0.700296,0.022914,0.218582
14,causal_shared,shared,"[(38, shared)]",1,always_active_shared,47,0.000010,3145728,19,10.884217,...,0.625509,0.109729,0.621286,0.636333,0.00,0.00,0.557243,0.698087,0.012969,0.207716
4,causal_shared,shared,"[(38, shared)]",1,always_active_shared,11,0.000010,3145728,19,10.945865,...,0.620382,0.103541,0.621286,0.633000,0.00,0.00,0.549717,0.694001,0.005822,0.202546
16,routing_routed,routed,"[(29, 194)]",1,forced_lowest_slot,11,0.000010,3145728,19,8.663449,...,0.793884,0.127669,0.632952,0.644667,0.00,0.00,0.704723,0.886660,-0.014744,0.267280
18,routing_routed,routed,"[(29, 194)]",1,forced_lowest_slot,23,0.000010,3145728,19,8.664234,...,0.792846,0.136529,0.626286,0.636333,0.00,0.00,0.703251,0.884454,-0.004513,0.269371
20,routing_routed,routed,"[(29, 194)]",1,forced_lowest_slot,47,0.000010,3145728,19,8.629311,...,0.786940,0.122708,0.637952,0.641333,0.00,0.00,0.701547,0.879116,-0.018344,0.261559
17,causal_routed,routed,"[(36, 229)]",1,forced_lowest_slot,23,0.000010,3145728,19,1.952222,...,0.023107,0.000617,0.621286,0.636333,0.00,0.00,0.013745,0.032707,-0.014139,0.015125
15,causal_routed,routed,"[(36, 229)]",1,forced_lowest_slot,11,0.000010,3145728,19,1.950006,...,0.022666,0.003905,0.621286,0.636333,0.00,0.00,0.011135,0.033869,-0.012593,0.019515
19,causal_routed,routed,"[(36, 229)]",1,forced_lowest_slot,47,0.000010,3145728,19,1.945202,...,0.021164,0.001852,0.621286,0.636333,0.00,0.00,0.012113,0.031221,-0.011227,0.015707
13,gradient_routed,routed,"[(25, 168)]",1,natural,47,0.000010,3145728,19,11.123634,...,1.107595,0.296867,0.637952,0.646333,0.00,0.00,0.979481,1.244761,0.119925,0.480829


In [55]:
deterministic_summary = (
    runs_df[
        ~runs_df["selector"].str.startswith(
            "random_"
        )
        & ~runs_df["selector"].str.contains(
            "lr_sensitivity"
        )
    ]
    .groupby([
        "selector",
        "module_type",
        "budget_k",
        "protocol",
        "lr",
    ])
    .agg(
        n_runs=("order_seed", "count"),
        target_improvement_mean=(
            "target_improvement",
            "mean",
        ),
        target_improvement_std=(
            "target_improvement",
            "std",
        ),
        control_improvement_mean=(
            "control_improvement",
            "mean",
        ),
        specific_gain_mean=(
            "specific_gain",
            "mean",
        ),
        specific_gain_std=(
            "specific_gain",
            "std",
        ),
        utility_score_mean=(
            "utility_score",
            "mean",
        ),
        target_token_acc_mean=(
            "target_token_acc",
            "mean",
        ),
        target_seq_exact_mean=(
            "target_seq_exact",
            "mean",
        ),
        parameter_delta_mean=(
            "relative_parameter_delta",
            "mean",
        ),
        mean_grad_norm=(
            "mean_grad_norm",
            "mean",
        ),
    )
    .reset_index()
)

display(
    deterministic_summary.sort_values(
        [
            "budget_k",
            "protocol",
            "target_improvement_mean",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

deterministic_summary.to_csv(
    RESULTS / "v6_deterministic_summary.csv",
    index=False,
)

,selector,module_type,budget_k,protocol,lr,n_runs,target_improvement_mean,target_improvement_std,control_improvement_mean,specific_gain_mean,specific_gain_std,utility_score_mean,target_token_acc_mean,target_seq_exact_mean,parameter_delta_mean,mean_grad_norm
3,causal_shared,shared,1,always_active_shared,0.00001,3,0.624143,0.003297,0.513023,0.111119,0.008361,0.624143,0.621286,0.00,0.005763,10.918800
9,routing_routed,routed,1,forced_lowest_slot,0.00001,3,0.791223,0.003746,0.662255,0.128969,0.007002,0.791223,0.632397,0.00,0.004732,8.652331
1,causal_routed,routed,1,forced_lowest_slot,0.00001,3,0.022312,0.001018,0.020187,0.002125,0.001661,0.022312,0.621286,0.00,0.004192,1.949143
5,gradient_routed,routed,1,natural,0.00001,3,1.097596,0.009654,0.800502,0.297094,0.002779,1.097596,0.637952,0.00,0.005550,11.158541
10,routing_routed,routed,1,natural,0.00001,3,0.794624,0.004666,0.666831,0.127793,0.007960,0.794624,0.636286,0.00,0.004780,8.602441
2,causal_routed,routed,1,natural,0.00001,3,0.031493,0.002957,0.029147,0.002346,0.003967,0.031493,0.621286,0.00,0.004097,1.052693
6,hybrid_causal_exposure,routed,1,natural,0.00001,3,0.031493,0.002957,0.029147,0.002346,0.003967,0.031493,0.621286,0.00,0.004097,1.052693
4,gradient_k4,routed,4,natural,0.00001,3,2.719969,0.001105,2.132641,0.587328,0.017037,2.719969,0.666286,0.08,0.004869,15.353121
8,routing_k4,routed,4,natural,0.00001,3,1.902243,0.033942,1.348427,0.553816,0.042848,1.902243,0.630571,0.00,0.004819,10.620659
0,causal_k4,routed,4,natural,0.00001,3,1.067177,0.007141,0.863467,0.203710,0.004896,1.067177,0.636286,0.00,0.004572,7.888755


In [56]:
random_df = runs_df[
    runs_df["selector"].str.startswith(
        "random_"
    )
].copy()

if not random_df.empty:
    random_df["random_family"] = np.where(
        random_df["selector"].str.startswith(
            "random_global"
        ),
        "random_global",
        "random_same_layer",
    )

    random_summary = (
        random_df.groupby(
            "random_family"
        )
        .agg(
            n=("selector", "count"),
            target_mean=(
                "target_improvement",
                "mean",
            ),
            target_std=(
                "target_improvement",
                "std",
            ),
            specific_mean=(
                "specific_gain",
                "mean",
            ),
            specific_std=(
                "specific_gain",
                "std",
            ),
        )
        .reset_index()
    )

    display(random_summary)

    random_summary.to_csv(
        RESULTS / "v6_random_summary.csv",
        index=False,
    )

,random_family,n,target_mean,target_std,specific_mean,specific_std
0,random_global,5,0.012474,0.024918,-0.005287,0.017622
1,random_same_layer,5,0.004264,0.002046,0.002399,0.003213


## Explicit old-result confound report

In [57]:
train_supervised = selector_exposure_df[
    (selector_exposure_df["dataset"] == "train_target")
    & (selector_exposure_df["scope"] == "supervised")
].copy()

print(
    "This table is the first thing to inspect before "
    "interpreting adaptation winners:"
)

display(
    train_supervised[
        [
            "selector",
            "layer",
            "expert",
            "selected_hits",
            "token_positions",
            "selected_rate",
            "routing_mass",
        ]
    ].sort_values(
        "routing_mass",
        ascending=False,
    )
)

print(
    "\nIf routing_routed has far more supervised exposure "
    "than causal_routed, v5 was not exposure-matched even "
    "though it was parameter-matched."
)

This table is the first thing to inspect before interpreting adaptation winners:


,selector,layer,expert,selected_hits,token_positions,selected_rate,routing_mass
2,causal_routed,36,229,41,169,0.242604,0.071095
22,hybrid_causal_exposure,36,229,41,169,0.242604,0.071095
32,gradient_routed,25,168,50,169,0.295858,0.065800
12,routing_routed,29,194,53,169,0.313609,0.065278



If routing_routed has far more supervised exposure than causal_routed, v5 was not exposure-matched even though it was parameter-matched.


## LR-sensitivity table

In [58]:
lr_df = runs_df[
    runs_df["selector"].str.contains(
        "lr_sensitivity"
    )
].copy()

if not lr_df.empty:
    display(
        lr_df[
            [
                "selector",
                "lr",
                "target_improvement",
                "control_improvement",
                "specific_gain",
                "target_token_acc",
                "target_seq_exact",
                "relative_parameter_delta",
            ]
        ].sort_values(
            [
                "selector",
                "lr",
            ]
        )
    )

,selector,lr,target_improvement,control_improvement,specific_gain,target_token_acc,target_seq_exact,relative_parameter_delta
43,causal_routed_lr_sensitivity,0.000003,0.013602,-0.000323,0.013925,0.621286,0.0,0.001238
45,causal_routed_lr_sensitivity,0.000030,0.086854,0.073074,0.013780,0.621286,0.0,0.011861
44,routing_routed_lr_sensitivity,0.000003,0.130233,0.114624,0.015609,0.621286,0.0,0.001456
46,routing_routed_lr_sensitivity,0.000030,2.143583,1.642663,0.500920,0.637952,0.0,0.013608


# Phase N — Paired held-out comparison: causal vs routing

In [59]:
def paired_method_bootstrap(
    nll_a,
    nll_b,
    mask,
    n_boot=20000,
    seed=999,
):
    # Positive means A improves more than B.
    # Base cancels: improvement_A - improvement_B = nll_B - nll_A.
    diff = (
        np.asarray(nll_b)[mask]
        - np.asarray(nll_a)[mask]
    )

    rng = np.random.default_rng(
        seed
    )

    vals = np.empty(n_boot)

    for i in range(n_boot):
        vals[i] = rng.choice(
            diff,
            size=len(diff),
            replace=True,
        ).mean()

    return {
        "mean_advantage_a_over_b": float(
            diff.mean()
        ),
        "ci_low": float(
            np.quantile(vals, 0.025)
        ),
        "ci_high": float(
            np.quantile(vals, 0.975)
        ),
        "p_a_gt_b": float(
            (vals > 0).mean()
        ),
    }

pairwise_rows = []

for protocol in [
    "natural",
    "forced_lowest_slot",
]:
    for seed in TRAIN_ORDER_SEEDS:
        key_c = (
            "causal_routed",
            1,
            protocol,
            seed,
            FIXED_LR,
        )
        key_r = (
            "routing_routed",
            1,
            protocol,
            seed,
            FIXED_LR,
        )

        if (
            key_c not in RUN_STATES
            or key_r not in RUN_STATES
        ):
            continue

        c = RUN_STATES[
            key_c
        ]["per_example_nll"]

        r = RUN_STATES[
            key_r
        ]["per_example_nll"]

        stats = paired_method_bootstrap(
            c,
            r,
            heldout_target_mask,
            seed=5000 + seed,
        )

        pairwise_rows.append({
            "protocol": protocol,
            "seed": seed,
            **stats,
        })

pairwise_df = pd.DataFrame(
    pairwise_rows
)

display(pairwise_df)

pairwise_df.to_csv(
    RESULTS / "v6_causal_vs_routing_paired_bootstrap.csv",
    index=False,
)

,protocol,seed,mean_advantage_a_over_b,ci_low,ci_high,p_a_gt_b
0,natural,11,-0.759316,-0.848999,-0.672200,0.0
1,natural,23,-0.765620,-0.853428,-0.683236,0.0
2,natural,47,-0.764461,-0.852020,-0.678378,0.0
3,forced_lowest_slot,11,-0.771220,-0.859393,-0.685572,0.0
4,forced_lowest_slot,23,-0.769739,-0.857555,-0.686270,0.0
5,forced_lowest_slot,47,-0.765776,-0.853184,-0.678841,0.0


# Phase O — Automatic interpretation

In [60]:
def mean_result(
    selector,
    budget_k,
    protocol,
):
    x = runs_df[
        (runs_df["selector"] == selector)
        & (runs_df["budget_k"] == budget_k)
        & (runs_df["protocol"] == protocol)
        & (runs_df["lr"] == FIXED_LR)
    ]

    if x.empty:
        return None

    return {
        "target": float(
            x["target_improvement"].mean()
        ),
        "specific": float(
            x["specific_gain"].mean()
        ),
    }

nat_c = mean_result(
    "causal_routed",
    1,
    "natural",
)
nat_r = mean_result(
    "routing_routed",
    1,
    "natural",
)
forced_c = mean_result(
    "causal_routed",
    1,
    "forced_lowest_slot",
)
forced_r = mean_result(
    "routing_routed",
    1,
    "forced_lowest_slot",
)
k4_c = mean_result(
    "causal_k4",
    4,
    "natural",
)
k4_r = mean_result(
    "routing_k4",
    4,
    "natural",
)
shared = mean_result(
    "causal_shared",
    1,
    "always_active_shared",
)

print("Natural K1 causal:", nat_c)
print("Natural K1 routing:", nat_r)
print("Forced K1 causal:", forced_c)
print("Forced K1 routing:", forced_r)
print("Natural K4 causal:", k4_c)
print("Natural K4 routing:", k4_r)
print("Causal shared:", shared)

print("\n=== Decision guide ===")

if (
    nat_c
    and nat_r
    and forced_c
    and forced_r
):
    natural_gap = (
        nat_r["target"]
        - nat_c["target"]
    )
    forced_gap = (
        forced_r["target"]
        - forced_c["target"]
    )

    print(
        "routing-causal target gap, natural:",
        natural_gap,
    )
    print(
        "routing-causal target gap, forced:",
        forced_gap,
    )

    if (
        natural_gap > 0
        and forced_gap
        < 0.5 * natural_gap
    ):
        print(
            "INTERPRETATION: much of routing's natural advantage "
            "is explained by gradient/routing access."
        )
    elif forced_gap > 0:
        print(
            "INTERPRETATION: routing remains more plastic even "
            "after forced access; necessity != plasticity."
        )

if (
    k4_c
    and nat_c
    and k4_c["target"]
    > nat_c["target"]
):
    print(
        "INTERPRETATION: causal adaptation improves with coalition "
        "budget; single-expert surgery was too narrow."
    )

if (
    shared
    and nat_c
    and shared["target"]
    > nat_c["target"]
):
    print(
        "INTERPRETATION: the always-active shared expert is a "
        "stronger same-size adaptation substrate than the top "
        "causal routed expert."
    )

Natural K1 causal: {'target': 0.03149255116780599, 'specific': 0.002346038818359375}
Natural K1 routing: {'target': 0.7946243286132812, 'specific': 0.1277929147084554}
Forced K1 causal: {'target': 0.022312005360921223, 'specific': 0.002124786376953125}
Forced K1 routing: {'target': 0.7912233670552572, 'specific': 0.12896855672200522}
Natural K4 causal: {'target': 1.067177136739095, 'specific': 0.20371023813883463}
Natural K4 routing: {'target': 1.9022427399953206, 'specific': 0.5538159211476644}
Causal shared: {'target': 0.6241426467895508, 'specific': 0.11111919085184734}

=== Decision guide ===
routing-causal target gap, natural: 0.7631317774454752
routing-causal target gap, forced: 0.7689113616943359
INTERPRETATION: routing remains more plastic even after forced access; necessity != plasticity.
INTERPRETATION: causal adaptation improves with coalition budget; single-expert surgery was too narrow.
INTERPRETATION: the always-active shared expert is a stronger same-size adaptation subs

# Phase P — Save complete v6 artifact bundle

In [62]:
import json
V6_DIR = RESULTS / "v6_falsification_grade"
V6_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tables_to_save = {
    "route_exposure_all_splits.csv": ROUTE_STATS,
    "selector_exposure.csv": selector_exposure_df,
    "shared_expert_causal_sweep.csv": shared_causal_df,
    "cross_method_causal_audit.csv": audit_df,
    "candidate_gradient_accessibility.csv": gradient_df,
    "training_runs.csv": runs_df,
    "training_history.csv": history_df,
    "deterministic_summary.csv": deterministic_summary,
    "paired_causal_vs_routing.csv": pairwise_df,
}

for filename, df in tables_to_save.items():
    df.to_csv(
        V6_DIR / filename,
        index=False,
    )

# Compact state banks + per-example NLL.
for key, payload in RUN_STATES.items():
    selector, k, protocol, seed, lr = key

    safe = (
        f"{selector}__K{k}__{protocol}"
        f"__seed{seed}__lr{lr:.0e}"
    ).replace("+", "")

    torch.save(
        payload["state_dict_cpu"],
        V6_DIR / f"{safe}.pt",
    )

    np.save(
        V6_DIR / f"{safe}_heldout_nll.npy",
        payload["per_example_nll"],
    )

manifest = {
    "model_id": MODEL_ID,
    "experiment_csv": str(
        EXPERIMENT_CSV
    ),
    "params_per_expert": int(
        params_per_expert
    ),
    "shared_params": int(
        shared_param_count
    ),
    "causal_seed_pair": list(
        CAUSAL_SEED_PAIR
    ),
    "causal_pair": list(
        CAUSAL_PAIR
    ),
    "routing_pair": list(
        ROUTING_PAIR
    ),
    "hybrid_pair": list(
        HYBRID_PAIR
    ),
    "gradient_pair": list(
        GRADIENT_PAIR
    ),
    "shared_causal_layer": int(
        SHARED_CAUSAL_LAYER
    ),
    "causal_k4": [
        list(x) for x in CAUSAL_K4
    ],
    "routing_k4": [
        list(x) for x in ROUTING_K4
    ],
    "hybrid_k4": [
        list(x) for x in HYBRID_K4
    ],
    "gradient_k4": [
        list(x) for x in GRADIENT_K4
    ],
    "fixed_lr": FIXED_LR,
    "training_order_seeds": (
        TRAIN_ORDER_SEEDS
    ),
    "forced_training_semantics": (
        "replace lowest-weight top-k expert ID with selected "
        "expert while preserving the displaced slot weight; "
        "normal routing restored for evaluation"
    ),
}

(
    V6_DIR / "manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

archive = shutil.make_archive(
    str(V6_DIR),
    "zip",
    root_dir=V6_DIR,
)

print("v6 results:", V6_DIR)
print("v6 archive:", archive)

v6 results: /home/ec2-user/workspace/laguna_xs2_v5_results/v6_falsification_grade
v6 archive: /home/ec2-user/workspace/laguna_xs2_v5_results/v6_falsification_grade.zip


# Final validity checklist

In [63]:
checks = {
    "same external CSV used for selection/train/heldout": True,
    "heldout unused for selector construction": True,
    "routing exposure measured on supervised train positions": True,
    "routed surgery pre-training equivalence tested": True,
    "shared surgery pre-training equivalence tested": True,
    "frozen base guarded after every arm": True,
    "same-size shared expert baseline included": True,
    "natural vs forced routing comparison included": True,
    "K1 vs K4 budget scaling included": True,
    "multiple order seeds included": len(TRAIN_ORDER_SEEDS) >= 3,
    "multiple random experts included": N_RANDOM_DRAWS >= 5,
    "LR sensitivity included": RUN_LR_SENSITIVITY,
    "signed target-specific gain reported": True,
    "paired heldout bootstrap included": True,
}

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-",
        name,
    )

if not all(checks.values()):
    raise RuntimeError(
        "One or more falsification-grade checks failed."
    )

print("\nV6 validity checklist: PASS")

PASS - same external CSV used for selection/train/heldout
PASS - heldout unused for selector construction
PASS - routing exposure measured on supervised train positions
PASS - routed surgery pre-training equivalence tested
PASS - shared surgery pre-training equivalence tested
PASS - frozen base guarded after every arm
PASS - same-size shared expert baseline included
PASS - natural vs forced routing comparison included
PASS - K1 vs K4 budget scaling included
PASS - multiple order seeds included
PASS - multiple random experts included
PASS - LR sensitivity included
PASS - signed target-specific gain reported
PASS - paired heldout bootstrap included

V6 validity checklist: PASS
